# Dataset Preparation for Non-Sequential Models  
## v8 Cascade Dataset — Annotated Readable Version

This notebook builds window-level datasets for early clinical deterioration prediction.

It supports two modelling patterns from the same generated dataset:

1. **First-deterioration prediction**  
   Use only windows before any deterioration has happened.

2. **Deterioration cascade prediction**  
   Keep later windows after earlier events, but only predict event types that have not happened yet.

This annotated version keeps the same core dataset logic, but adds more explanations so readers with little or no Python background can understand the pipeline.

---

## Key idea in plain language

For each patient case, the notebook repeatedly asks:

> “Using the patient information and vital signs available up to this time point, can we predict whether deterioration will happen in the next few hours?”

Each prediction row is one **time window** for one patient.

Example:

- Observation window: previous 6 hours of vital signs
- Prediction horizon: next 4, 6, or 12 hours
- Target `y_o2`: whether oxygen escalation happens in that future horizon

---

## Important leakage rule

The model must only use information available **before `window_end`**.

The notebook uses future event times only to create labels, then removes internal event-time columns from the final model-ready CSV files.


# Reading guide: important concepts

| Term | Meaning |
|---|---|
| `case_no_deid_clean` | Anonymous patient/case identifier |
| `window_start` / `window_end` | Time interval used to extract past information |
| `horizon_end` | End of the future prediction period |
| `m_hours` | Observation-window length, e.g. 4h, 6h, 12h, 24h |
| `h_hours` | Prediction-horizon length, e.g. 4h, 6h, 12h |
| `label_*` | Case-level outcome: whether the event ever happened in this case |
| `y_*` | Window-level target: whether the event happens in this window’s future horizon |
| `eligible_y_*` | Whether this window is still valid for predicting that target event |
| `prior_*` | Whether an event already happened before `window_end` |
| `time_since_prior_*` | How long ago a prior event happened, measured from `window_end` |
| `feature base` | Expensive-to-build window features before horizon-specific labels are attached |

The most important distinction is:

- `label_o2` = this case ever had O2 increase.
- `y_o2` = O2 increase happens within the current row’s future horizon.


# Pipeline overview

The notebook follows this sequence:

1. Load prepared patient, vital-sign, diagnosis, and deterioration files.
2. Clean case IDs so that tables can be merged safely.
3. Create case-level labels and event-time columns.
4. Select cases that have all required data sources.
5. Split cases into train / validation / test.
6. Create grouped secondary-diagnosis features.
7. Convert charted and sensor vitals into one long time-series format.
8. Merge charted and sensor signals into unified vital-sign names.
9. Build reusable window feature bases for each observation length `m`.
10. Add cascade labels for each prediction horizon `h`.
11. Save train / validation / test CSV files.
12. Save summary tables for checking sample size, positive rates, missingness, and censoring.
13. Provide helper functions for cascade and first-deterioration modelling.

Runtime note: the expensive step is building reusable feature bases. Labelling multiple horizons is much faster.



# Extra non-technical explanation: how to read this notebook

This notebook has two purposes:

1. **Generate window-level datasets** for early deterioration prediction.
2. **Keep the logic understandable and auditable**, so the dataset design can be explained to supervisors and non-technical readers.

The code may look long because the notebook is careful about time logic and leakage control. The core idea is still simple:

- Look at a patient's past vital signs within an observation window.
- Use only information available before the prediction time.
- Predict whether deterioration happens in the next few hours.
- For cascade modelling, allow earlier deterioration events to become valid prior-history features.

No model is trained in this notebook. This notebook only creates clean datasets for later modelling.



# Leakage-control principles used in this notebook

A dataset has leakage if it gives the model information that would not be available at prediction time. This notebook tries to avoid leakage using these rules:

- Future event timestamps are used only to create labels, then removed from final modelling files.
- Primary diagnosis is excluded because it may be finalized after admission or discharge.
- Secondary diagnosis is used because project background suggests it is mostly available before W36 admission.
- A target event is not predicted after that same event has already happened.
- Prediction windows are kept only if the full prediction horizon is observable inside the W36 period.
- Train/validation/test split is done by case, not by window.

These rules do not make the dataset mathematically perfect, but they make the design clinically and methodologically defensible.



# Important terminology: `label`, `y`, `eligible`, and `prior`

These terms appear many times in the notebook:

| Term | Meaning | Example |
|---|---|---|
| `label_*` | Case-level outcome: whether the case ever had the event | `label_o2_increase = 1` means the patient had O2 increase sometime |
| `y_*` | Window-level target: whether the event happens in the next prediction horizon | `y_o2 = 1` means O2 happens after this window end and before horizon end |
| `eligible_y_*` | Whether this window is valid for predicting this target | `eligible_y_o2 = 0` means O2 already happened before this window |
| `prior_*` | Whether an event already happened before prediction time | `prior_met = 1` means MET already happened before this window end |

For modelling, the final target should be a `y_*` column, not a `label_*` column.



# Cascade modelling in plain language

Earlier notebook versions focused on the first deterioration event. Once any deterioration happened, later windows were removed.

Version 8 is more flexible. It keeps later windows and records what has already happened before the prediction time.

Example:

- O2 increase happened at 12:00.
- A window ends at 14:00.
- The model is not allowed to predict O2 again, because O2 already happened.
- But the model can still predict MET activation, HD/ICU transfer, or death if those events have not happened yet.

This is how v8 supports deterioration-cascade modelling without using future information as predictors.


## Cell 1 explanation — paths and switches

This cell connects to Google Drive, defines input/output file paths, and controls optional behaviours.

Important switches:

- `RUN_EXAMPLE_DATASET`: if `False`, skip the example dataset build to save time.
- Output folder: stores final v8 datasets and summary tables.

The notebook should be run in Google Colab because it expects files under `/content/drive/My Drive/...`.


## Cell 1: Mount Google Drive, define file paths, and set run switches

### Non-technical note for Cell 1 — setup
This cell prepares the working environment. It connects Colab to Google Drive, defines where the source files live, and chooses where the new v8 dataset will be saved.

The important idea is simple: **all later cells use these paths and switches**. If a path is wrong here, later cells may fail even if the logic is correct. The switches are also useful because they let you test a small part of the notebook before running the whole long dataset-generation process.

The most important switch is `RUN_EXAMPLE_DATASET`. It is set to `False` by default so the notebook does not waste time generating an extra example dataset before the full run.

In [1]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# ============================================================
# Cell 1: Setup and path configuration
# ============================================================
# This cell prepares the notebook environment.
# It does not create any dataset yet.
#
# Main tasks:
#   1. Mount Google Drive.
#   2. Define where input files are stored.
#   3. Define where output datasets will be saved.
#   4. Set switches for optional example runs.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

# Base folder containing the prepared merged CSV files.
base_dir = "/content/drive/My Drive/merged_240920_240822_prepared"

# Input file paths.
case_master_path = f"{base_dir}/case_master_all_240920_240822.csv"
charted_vitals_path = f"{base_dir}/charted_vitals_all_240920_240822.csv"
diagnosis_summary_path = f"{base_dir}/diagnosis_summary_all_240920_240822.csv"
sensor_timeseries_path = f"{base_dir}/sensor_timeseries_all_240920_240822.csv"
cohort_summary_path = f"{base_dir}/cohort_summary_all_240920_240822.csv"

# Diagnosis policy for this version:
#   Use only diagnosis_summary_all_240920_240822.csv.
#   Secondary diagnoses are read from secondary_diagnosis_list,
#   where different diagnoses are separated by "|".
#   Primary diagnosis is not used as a prediction feature because it may
#   contain information assigned after W36 admission or discharge.

# Output folder for the v8 cascade prepared non-sequential datasets.
# A new folder is used so earlier dataset versions remain unchanged.
output_dir = f"{base_dir}/nonsequential_window_dataset_v8_cascade"
os.makedirs(output_dir, exist_ok=True)

# Switch for quick debugging.
# Keep this False for the full production run, because even one example
# dataset can take time to build.
RUN_EXAMPLE_DATASET = False

print("V8 cascade dataset output folder created:")
print(output_dir)
print("\nExample dataset building switch:", RUN_EXAMPLE_DATASET)

Mounted at /content/drive
V8 cascade dataset output folder created:
/content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade

Example dataset building switch: False


## Cell 2 explanation — load source files

This cell loads the prepared data tables used for dataset creation.

The important principle is that all tables must use the same cleaned case ID format. Otherwise, one patient may appear to be missing simply because the ID has a different text format, such as `123.0` vs `123`.


## Cell 2: Load the five prepared datasets

### Non-technical note for Cell 2 — loading data
This cell reads the prepared CSV files into memory. Each file contains one part of the patient story: patient/case information, charted vital signs, diagnosis summary, sensor vital signs, and cohort/source availability.

The cell also standardizes `case_no_deid_clean`. This is important because patient IDs can look slightly different across files, such as `12345`, `12345.0`, or ` 12345 `. If IDs are not cleaned consistently, the merge may silently lose patients.

In [14]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# ============================================================
# Cell 2: Load the prepared project data
# ============================================================
# Each file represents one source of information.
# Later cells combine them at case level and window level.
#
# Important safety point:
#   We standardize case IDs after loading, so that the same patient/case
#   can be matched across all files.
# ============================================================

case_master = pd.read_csv(case_master_path)
charted_vitals = pd.read_csv(charted_vitals_path)
diagnosis_summary = pd.read_csv(diagnosis_summary_path)
sensor_timeseries = pd.read_csv(sensor_timeseries_path)
cohort_summary = pd.read_csv(cohort_summary_path)


def clean_case_id_series(series):
    """
    Standardize case IDs before any filtering or merging.

    Why this matters:
        The same case ID can sometimes be read as "123", "123.0", or
        with extra spaces. If not standardized, valid cases may fail to
        match across tables.
    """

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    cleaned = cleaned.replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan,
        "NaN": np.nan
    })

    return cleaned


def ensure_case_id_column(df, table_name):
    """
    Make sure every loaded table has a clean case_no_deid_clean column.

    The function first looks for case_no_deid_clean. If not available,
    it tries several common case-ID column names.
    """

    possible_case_id_cols = [
        "case_no_deid_clean",
        "Case No DEID",
        "Case No DeID",
        "case_no_deid",
        "Case_No_DEID",
        "CASE_NO_DEID"
    ]

    source_col = None
    for col in possible_case_id_cols:
        if col in df.columns:
            source_col = col
            break

    if source_col is None:
        print(f"Warning: no case ID column found in {table_name}. Columns unchanged.")
        return df

    df = df.copy()
    df["case_no_deid_clean"] = clean_case_id_series(df[source_col])

    print(
        f"{table_name}: standardized case_no_deid_clean "
        f"from column '{source_col}'. Unique cases = "
        f"{df['case_no_deid_clean'].nunique()}"
    )

    return df


# Standardize case IDs in all loaded tables before downstream processing.
case_master = ensure_case_id_column(case_master, "case_master")
charted_vitals = ensure_case_id_column(charted_vitals, "charted_vitals")
diagnosis_summary = ensure_case_id_column(diagnosis_summary, "diagnosis_summary")
sensor_timeseries = ensure_case_id_column(sensor_timeseries, "sensor_timeseries")
cohort_summary = ensure_case_id_column(cohort_summary, "cohort_summary")

print("\nLoaded datasets:")
print("case_master:", case_master.shape)
print("charted_vitals:", charted_vitals.shape)
print("diagnosis_summary:", diagnosis_summary.shape)
print("sensor_timeseries:", sensor_timeseries.shape)
print("cohort_summary:", cohort_summary.shape)

case_master: standardized case_no_deid_clean from column 'case_no_deid_clean'. Unique cases = 1058
charted_vitals: standardized case_no_deid_clean from column 'case_no_deid_clean'. Unique cases = 1058
diagnosis_summary: standardized case_no_deid_clean from column 'case_no_deid_clean'. Unique cases = 1054
sensor_timeseries: standardized case_no_deid_clean from column 'case_no_deid_clean'. Unique cases = 443

Loaded datasets:
case_master: (1058, 65)
charted_vitals: (372880, 9)
diagnosis_summary: (1054, 12)
sensor_timeseries: (279672, 21)
cohort_summary: (18, 3)


## Time conversion explanation

Clinical event times and vital-sign times must be real datetime values, not plain text.

This cell converts important timestamp columns into datetime format so the notebook can correctly compare times, such as:

- Did an event happen before `window_end`?
- Did an event happen inside `[window_end, horizon_end)`?
- How many hours since a prior event?


## Convert important time columns into datetime format

### Non-technical note for Cell 3 — time conversion
This cell converts date/time columns into real Python datetime objects. This is necessary because the whole project depends on time logic: observation windows, prediction horizons, event times, and movement end time.

Without this conversion, the code might treat dates as normal text and compare them incorrectly. For example, the notebook needs to know whether an O2 increase happened before or after a window end time.

In [15]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# Time columns in case-level master table
case_time_cols = [
    "W36 Start DT_dt",
    "Movement End DT_dt",
    "O2 increased provision DT_dt",
    "Increased IV DT_dt",
    "MET activation after W36 Admission_dt",
    "Death Date_dt",
    "next hd/icu admission dt_dt",
    "o2_increase_event_dt"
]

for col in case_time_cols:
    if col in case_master.columns:
        case_master[col] = pd.to_datetime(case_master[col], errors="coerce")

# Time column in charted vital signs
charted_vitals["created_datetime_dt"] = pd.to_datetime(
    charted_vitals["created_datetime_dt"],
    errors="coerce"
)

# Time column in sensor vital signs
sensor_timeseries["sensor_datetime_dt"] = pd.to_datetime(
    sensor_timeseries["sensor_datetime_dt"],
    errors="coerce"
)

print("Datetime conversion finished.")

Datetime conversion finished.


## Cell 4 explanation — case-level labels

This cell creates case-level outcome indicators, such as whether a case ever had O2 increase, MET activation, or death.

These are not the final model targets. They are used to summarize outcomes and support window-level label creation later.

The final model targets are `y_*` columns created later for each prediction window.


## Cell 4: Define case-level labels for deterioration outcomes

### Non-technical note for Cell 4 — case-level labels
This cell creates patient-level outcome indicators. These variables answer: **Did this case ever have this event during the dataset period?**

These are not the final modelling targets. They are used as the starting point to create window-level targets later. In the final model, we predict `y_*` columns, not `label_*` columns.

A useful distinction:
- `label_o2_increase`: this patient ever had O2 increase.
- `y_o2`: in this specific prediction window, will O2 increase happen in the next horizon?

In [16]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

df_cases = case_master.copy()

# -------------------------------
# 1. HD / ICU transfer label
# -------------------------------
df_cases["label_hd_icu"] = (
    (df_cases["HD/ICU"] == 1) |
    (df_cases["next hd/icu admission dt_dt"].notna())
).astype(int)

# -------------------------------
# 2. O2 increase label
# -------------------------------
# Important:
#   Earlier data checks suggested that O2 timestamp fields may exist even
#   for some cases that are not confirmed O2-increase events.
#   Therefore, when o2_increase_confirmed exists, we trust the confirmation
#   flag first and do not use timestamp alone to define a positive case.
if "o2_increase_confirmed" in df_cases.columns:
    o2_confirmed = (
        pd.to_numeric(df_cases["o2_increase_confirmed"], errors="coerce")
        .fillna(0)
        .eq(1)
    )
    df_cases["label_o2_increase"] = o2_confirmed.astype(int)
else:
    # Fallback only if the confirmation column is not available.
    df_cases["label_o2_increase"] = df_cases["o2_increase_event_dt"].notna().astype(int)

# -------------------------------
# 3. IV increase label
# -------------------------------
df_cases["label_iv_increase"] = (
    df_cases["Increased IV DT_dt"].notna()
).astype(int)

# -------------------------------
# 4. MET activation label
# -------------------------------
df_cases["label_met"] = (
    df_cases["MET activation after W36 Admission_dt"].notna()
).astype(int)

# -------------------------------
# 5. Death label
# -------------------------------
df_cases["label_death"] = (
    (df_cases["Death 1M From W36 Admission"] == 1) |
    (df_cases["Death 3M From W36 Admission"] == 1) |
    (df_cases["Death Date_dt"].notna())
).astype(int)

# -------------------------------
# Composite case-level label
# -------------------------------
# This means: did this patient experience any deterioration event?
df_cases["label_any_deterioration"] = (
    (df_cases["label_hd_icu"] == 1) |
    (df_cases["label_o2_increase"] == 1) |
    (df_cases["label_iv_increase"] == 1) |
    (df_cases["label_met"] == 1) |
    (df_cases["label_death"] == 1)
).astype(int)

label_cols = [
    "label_hd_icu",
    "label_o2_increase",
    "label_iv_increase",
    "label_met",
    "label_death",
    "label_any_deterioration"
]

print("Case-level label counts:")
print(df_cases[label_cols].sum())

if "o2_increase_confirmed" in df_cases.columns:
    print("\nO2 label check:")
    print("Confirmed O2-increase cases:", int(df_cases["label_o2_increase"].sum()))
    print("Rows with non-missing O2 timestamp:", int(df_cases["o2_increase_event_dt"].notna().sum()))
    print("Rows with timestamp but not confirmed:", int(((df_cases["o2_increase_event_dt"].notna()) & (df_cases["label_o2_increase"] == 0)).sum()))

Case-level label counts:
label_hd_icu               136
label_o2_increase           34
label_iv_increase          124
label_met                   13
label_death                193
label_any_deterioration    385
dtype: int64

O2 label check:
Confirmed O2-increase cases: 34
Rows with non-missing O2 timestamp: 34
Rows with timestamp but not confirmed: 0


## Cell 5 explanation — event-time columns

Window-level prediction needs event timestamps.

A case-level label only says “this event happened sometime.”  
An event-time column says “this event happened at this specific time.”

The notebook uses event-time columns to decide whether an event falls into the future prediction horizon of each window.


## Cell 5: Define the first timed deterioration event

### Non-technical note for Cell 5 — event times
This cell identifies the actual time of each deterioration event. This is required because a sliding-window dataset needs to know whether an event happened inside the prediction horizon.

The O2 logic is deliberately cautious. If a confirmed O2-increase indicator exists, the code only accepts O2 event time when the case is confirmed positive. This avoids treating unrelated O2 timestamps as true deterioration events.

In [17]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# These event times are used for window-level labels.
# Important:
# - A case-level label can be positive even without a timestamp.
# - But for sliding-window prediction, we can only label a window
#   as positive if the event time is known.

# HD/ICU, IV, MET, and Death event times come directly from timestamp columns.
df_cases["event_time_hd_icu"] = df_cases["next hd/icu admission dt_dt"]
df_cases["event_time_iv"] = df_cases["Increased IV DT_dt"]
df_cases["event_time_met"] = df_cases["MET activation after W36 Admission_dt"]
df_cases["event_time_death"] = df_cases["Death Date_dt"]

# O2 event time uses the confirmed O2 label.
# If a timestamp exists but the case is not confirmed as O2 increase,
# the timestamp is set to missing to avoid false positive windows.
df_cases["event_time_o2"] = pd.NaT
df_cases.loc[
    df_cases["label_o2_increase"] == 1,
    "event_time_o2"
] = df_cases.loc[
    df_cases["label_o2_increase"] == 1,
    "o2_increase_event_dt"
]

event_time_cols = [
    "event_time_hd_icu",
    "event_time_o2",
    "event_time_iv",
    "event_time_met",
    "event_time_death"
]

for col in event_time_cols:
    df_cases[col] = pd.to_datetime(df_cases[col], errors="coerce")

# Earliest timed deterioration event among all five outcomes.
# This is used to avoid building windows after deterioration already happened.
df_cases["first_event_time"] = df_cases[event_time_cols].min(axis=1)

# Indicator: whether this case has at least one usable event timestamp.
df_cases["has_timed_event"] = df_cases["first_event_time"].notna().astype(int)

print("Cases with any deterioration label:")
print(df_cases["label_any_deterioration"].sum())

print("\nCases with usable timed event:")
print(df_cases["has_timed_event"].sum())

print("\nTimed event counts:")
print(df_cases[event_time_cols].notna().sum())

Cases with any deterioration label:
385

Cases with usable timed event:
298

Timed event counts:
event_time_hd_icu     24
event_time_o2         34
event_time_iv        124
event_time_met        13
event_time_death     193
dtype: int64


## Cell 6 explanation — full-source cohort

This step keeps only cases that are present across the required data sources.

This makes the modelling cohort consistent and avoids comparing patients with very different source availability.


## Cell 6: Select the full-source cohort

### Non-technical note for Cell 6 — full-source cohort
This cell keeps only cases that have all required data sources. This follows the project rule: a case should be included only if it is present across the key datasets needed for modelling.

This improves consistency. It means every included case has the required patient information, diagnosis summary, charted vitals, sensor vitals, and usable W36 time interval.

In [18]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# For this first version, we use:
# patient data + diagnosis + charted vitals + Respiree sensor data

full_cohort = df_cases[
    (df_cases["has_ehints_data"] == 1) &
    (df_cases["has_diagnosis"] == 1) &
    (df_cases["has_charted_vitals"] == 1) &
    (df_cases["has_respiree_sensor"] == 1)
].copy()

# Keep only cases with valid start and end time
full_cohort = full_cohort[
    full_cohort["W36 Start DT_dt"].notna() &
    full_cohort["Movement End DT_dt"].notna()
].copy()

# Remove cases where end time is not after start time
full_cohort = full_cohort[
    full_cohort["Movement End DT_dt"] > full_cohort["W36 Start DT_dt"]
].copy()

print("Full-source cohort size:")
print(full_cohort.shape)

print("Case-level deterioration distribution:")
print(full_cohort["label_any_deterioration"].value_counts())

Full-source cohort size:
(441, 78)
Case-level deterioration distribution:
label_any_deterioration
0    251
1    190
Name: count, dtype: int64


## Cell 7 explanation — case-level split

Train / validation / test split is done by case, not by window.

This is essential because one case creates many window rows. If windows from the same case appeared in both train and test, the test performance would be over-optimistic.


## Cell 7: Split cases into train / validation / test

### Non-technical note for Cell 7 — case-level train/validation/test split
This cell splits patients into train, validation, and test sets. The split is done by case, not by window.

This is critical. If windows from the same patient appeared in both train and test, the model could partially memorize patient-specific patterns and look better than it really is. Case-level splitting prevents that leakage.

In [19]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split

# We split by case first.
# This prevents windows from the same patient appearing in both train and test.

case_ids = full_cohort["case_no_deid_clean"]
case_labels = full_cohort["label_any_deterioration"]

# First split: 70% train, 30% temporary
train_ids, temp_ids = train_test_split(
    case_ids,
    test_size=0.30,
    random_state=42,
    stratify=case_labels
)

# Prepare labels for temporary cases
temp_labels = full_cohort.set_index("case_no_deid_clean").loc[temp_ids, "label_any_deterioration"]

# Second split: temporary 30% becomes 15% validation and 15% test
val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

# Add split label to full cohort
full_cohort["data_split"] = "unused"
full_cohort.loc[full_cohort["case_no_deid_clean"].isin(train_ids), "data_split"] = "train"
full_cohort.loc[full_cohort["case_no_deid_clean"].isin(val_ids), "data_split"] = "validation"
full_cohort.loc[full_cohort["case_no_deid_clean"].isin(test_ids), "data_split"] = "test"

print("Split size by cases:")
print(full_cohort["data_split"].value_counts())

print("\nDeterioration rate by split:")
print(full_cohort.groupby("data_split")["label_any_deterioration"].mean())

Split size by cases:
data_split
train         308
test           67
validation     66
Name: count, dtype: int64

Deterioration rate by split:
data_split
test          0.432836
train         0.431818
validation    0.424242
Name: label_any_deterioration, dtype: float64


## Cell 8A explanation — grouped secondary diagnosis features

This section uses secondary diagnosis only.

Project assumption:

- Secondary diagnoses were mostly made before W36 ward admission, so they can represent baseline risk.
- Primary diagnosis may contain leakage because it can be finalized after admission or discharge, so it is not used as a prediction feature.

Instead of creating hundreds of sparse raw diagnosis columns, this cell creates clinically meaningful grouped flags, such as renal disease, pneumonia, infection/sepsis, cardiac disease, and frailty-related diagnosis.


## Cell 8A: Create secondary-diagnosis group features

This section uses **secondary diagnosis only**. Primary diagnosis is intentionally excluded from prediction features because it may be assigned after W36 admission or even after discharge, which can cause data leakage.


### Non-technical note for Cell 8A — secondary diagnosis grouping
This is one of the most important feature-engineering cells. It uses only the `secondary_diagnosis_list` column from the diagnosis summary file. Primary diagnosis is intentionally excluded because it may contain leakage.

The cell does three things:
1. Splits the secondary-diagnosis list into separate diagnoses using `|`.
2. Groups diagnosis text into clinically meaningful categories, such as renal disease, pneumonia, sepsis, cardiac disease, and frailty.
3. Merges these grouped diagnosis flags back into the patient-level cohort.

The cell is also safe to rerun. Before merging secondary-diagnosis columns again, it removes previous versions of those columns from `full_cohort`. This prevents duplicate `_x` and `_y` columns and fixes the earlier `secondary_diagnosis_count` KeyError.

In [20]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# ============================================================
# Cell 8A: Create grouped secondary-diagnosis features
# ============================================================
# Goal:
#   Turn the pipe-separated secondary diagnosis list into a small number
#   of clinically meaningful risk groups.
#
# Why secondary diagnosis only?
#   Project background suggests most secondary diagnoses were made before
#   W36 ward admission. Primary diagnosis may be finalized later, so it is
#   NOT used as a model feature in this notebook.
#
# Input used in this version:
#   diagnosis_summary_all_240920_240822.csv
#   column: secondary_diagnosis_list
#   separator: |
#
# Output:
#   secondary_dx_features: one row per case, with grouped dummy variables.
# ============================================================

import re

# ------------------------------------------------------------
# 1. Load secondary diagnosis list from the merged diagnosis summary
# ------------------------------------------------------------
# We intentionally do not load raw 240920/240822 Excel files here.
# This avoids accidental duplication when both raw and merged diagnosis
# files exist in Google Drive.

required_diag_cols = ["case_no_deid_clean", "secondary_diagnosis_list"]
missing_diag_cols = [col for col in required_diag_cols if col not in diagnosis_summary.columns]

if len(missing_diag_cols) > 0:
    raise KeyError(
        "diagnosis_summary_all_240920_240822.csv must contain these columns: "
        f"{required_diag_cols}. Missing columns: {missing_diag_cols}. "
        "If the summary file is incomplete, rebuild it from the raw eHINTS diagnosis files first."
    )

secondary_dx_source = diagnosis_summary[required_diag_cols].copy()
secondary_dx_source = secondary_dx_source.drop_duplicates()

secondary_dx_source["case_no_deid_clean"] = (
    secondary_dx_source["case_no_deid_clean"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Split one pipe-separated diagnosis list into multiple rows.
secondary_dx_source["secondary_diagnosis_list"] = (
    secondary_dx_source["secondary_diagnosis_list"]
    .fillna("")
    .astype(str)
)

secondary_dx_raw = (
    secondary_dx_source
    .assign(
        secondary_diagnosis_description=lambda d: d["secondary_diagnosis_list"].str.split("|")
    )
    .explode("secondary_diagnosis_description")
    [["case_no_deid_clean", "secondary_diagnosis_description"]]
    .copy()
)

secondary_dx_raw["secondary_diagnosis_description"] = (
    secondary_dx_raw["secondary_diagnosis_description"]
    .astype(str)
    .str.strip()
)

# Remove empty or placeholder diagnosis rows.
secondary_dx_raw = secondary_dx_raw[
    ~secondary_dx_raw["secondary_diagnosis_description"].isin([
        "", "nan", "None", "**********"
    ])
].copy()

# Remove duplicate diagnosis entries for the same case.
# This protects secondary_diagnosis_count and group summaries.
secondary_dx_raw = secondary_dx_raw.drop_duplicates(
    subset=["case_no_deid_clean", "secondary_diagnosis_description"]
).copy()


def normalize_dx_text(text):
    """Lowercase and remove punctuation for easier keyword matching."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


secondary_dx_raw["secondary_dx_clean_text"] = (
    secondary_dx_raw["secondary_diagnosis_description"]
    .apply(normalize_dx_text)
)

print("Secondary diagnosis source:")
print(diagnosis_summary_path)
print("\nCleaned secondary diagnosis rows:", secondary_dx_raw.shape[0])
print("Unique cases with secondary diagnosis:", secondary_dx_raw["case_no_deid_clean"].nunique())


# ------------------------------------------------------------
# 2. Define clinically meaningful secondary diagnosis groups
# ------------------------------------------------------------
# These are grouped risk markers, not raw one-hot diagnosis codes.
# A case can belong to more than one group.

secondary_dx_group_keywords = {
    "secondary_dx_infection_sepsis": [
        "sepsis", "septic", "bacteraemia", "bacteremia",
        "infection", "infectious", "urinary tract infection",
        "e coli", "escherichia coli"
    ],

    "secondary_dx_pneumonia": [
        "pneumonia", "bronchopneumonia"
    ],

    "secondary_dx_respiratory": [
        "respiratory failure", "respiratory insufficiency",
        "chronic obstructive pulmonary", "copd", "asthma",
        "bronchitis", "bronchiectasis", "hypoxaemia", "hypoxemia",
        "hypoxia", "pleural effusion", "pulmonary oedema",
        "pulmonary edema", "atelectasis"
    ],

    "secondary_dx_cardiac": [
        "heart failure", "cardiac failure", "ischaemic heart disease",
        "ischemic heart disease", "myocardial infarction",
        "atrial fibrillation", "atrial flutter", "arrhythmia",
        "cardiomyopathy", "valve", "valvular", "coronary", "angina"
    ],

    "secondary_dx_thrombosis_embolism": [
        "pulmonary embolism", "embolism", "thrombosis",
        "deep vein thrombosis", "dvt"
    ],

    "secondary_dx_renal_any": [
        "kidney", "renal", "nephropathy", "dialysis"
    ],

    "secondary_dx_acute_kidney_failure": [
        "acute kidney failure", "acute renal failure",
        "acute kidney injury", "aki"
    ],

    "secondary_dx_ckd_stage_3_5_or_dialysis": [
        "chronic kidney disease stage 3", "chronic kidney disease stage 4",
        "chronic kidney disease stage 5", "ckd stage 3",
        "ckd stage 4", "ckd stage 5", "dialysis"
    ],

    "secondary_dx_diabetes_complication": [
        "diabetes mellitus with", "diabetic nephropathy",
        "diabetic neuropathy", "diabetic retinopathy",
        "diabetic angiopathy", "diabetes mellitus with poor control",
        "diabetes mellitus with hypoglycaemia",
        "diabetes mellitus with hypoglycemia", "insulin resistance"
    ],

    "secondary_dx_hypertension": [
        "hypertension", "hypertensive"
    ],

    "secondary_dx_electrolyte_acid_base_volume": [
        "acidosis", "alkalosis", "hyponatraemia", "hyponatremia",
        "hypernatraemia", "hypernatremia", "hypokalaemia",
        "hypokalemia", "hyperkalaemia", "hyperkalemia",
        "hypo osmolality", "dehydration", "volume depletion",
        "fluid overload"
    ],

    "secondary_dx_anemia_bleeding_coagulation": [
        "anaemia", "anemia", "haemorrhage", "hemorrhage",
        "bleeding", "coagulation", "thrombocytopenia",
        "pancytopenia", "neutropenia", "iron deficiency"
    ],

    "secondary_dx_malignancy": [
        "malignant", "carcinoma", "neoplasm", "cancer",
        "lymphoma", "leukaemia", "leukemia", "tumour", "tumor"
    ],

    "secondary_dx_metastatic_cancer": [
        "metastatic", "metastasis", "secondary malignant",
        "malignant neoplasm of lymph nodes",
        "secondary and unspecified malignant"
    ],

    "secondary_dx_neuro_delirium_stroke": [
        "delirium", "dementia", "stroke", "cerebral infarction",
        "intracerebral haemorrhage", "intracerebral hemorrhage",
        "parkinson", "epilepsy", "seizure", "hemiplegia",
        "neurological"
    ],

    "secondary_dx_liver_cirrhosis_failure": [
        "cirrhosis", "hepatic failure", "liver failure",
        "chronic liver disease", "portal hypertension"
    ],

    "secondary_dx_gi_acute": [
        "gastrointestinal haemorrhage", "gastrointestinal hemorrhage",
        "intestinal obstruction", "perforation", "peritonitis",
        "ileus", "melena", "haematemesis", "hematemesis"
    ],

    "secondary_dx_frailty_pressure_ulcer_dysphagia_fall": [
        "decubitus ulcer", "pressure area", "pressure ulcer",
        "dysphagia", "fall", "frailty", "malnutrition",
        "cachexia", "difficulty in walking"
    ],

    "secondary_dx_hypotension_shock": [
        "hypotension", "shock", "hypovolaemic shock",
        "hypovolemic shock", "cardiogenic shock", "septic shock"
    ]
}

secondary_dx_group_cols = list(secondary_dx_group_keywords.keys())


def contains_any_keyword(text, keyword_list):
    """Return 1 if the diagnosis text contains any keyword in a group."""
    text = str(text)
    return int(any(keyword in text for keyword in keyword_list))


for group_name, keyword_list in secondary_dx_group_keywords.items():
    secondary_dx_raw[group_name] = secondary_dx_raw["secondary_dx_clean_text"].apply(
        lambda x: contains_any_keyword(x, keyword_list)
    )


# ------------------------------------------------------------
# 3. Aggregate diagnosis groups to case level
# ------------------------------------------------------------
# Each final row means: this case has / does not have this secondary
# diagnosis group at baseline.

secondary_dx_features = (
    secondary_dx_raw
    .groupby("case_no_deid_clean")
    .agg(
        secondary_diagnosis_count=("secondary_diagnosis_description", "nunique"),
        **{col: (col, "max") for col in secondary_dx_group_cols}
    )
    .reset_index()
)

secondary_dx_features["secondary_dx_severe_group_count"] = (
    secondary_dx_features[secondary_dx_group_cols].sum(axis=1)
)

# Add rows with all-zero diagnosis features for full-cohort cases that have
# no secondary diagnosis row after cleaning. This keeps the merge explicit.
all_full_cases = pd.DataFrame({
    "case_no_deid_clean": full_cohort["case_no_deid_clean"].astype(str).str.strip().unique()
})

secondary_dx_features = all_full_cases.merge(
    secondary_dx_features,
    on="case_no_deid_clean",
    how="left"
)

secondary_dx_fill_cols = [
    "secondary_diagnosis_count",
    "secondary_dx_severe_group_count"
] + secondary_dx_group_cols

for col in secondary_dx_fill_cols:
    secondary_dx_features[col] = secondary_dx_features[col].fillna(0).astype(int)

print("\nSecondary diagnosis grouped feature table:")
print(secondary_dx_features.shape)
display(secondary_dx_features.head())


# ------------------------------------------------------------
# 4. Save diagnosis checking tables
# ------------------------------------------------------------
# The group summary includes both prevalence and outcome rates.
# This helps us see which secondary diagnosis groups are common and which
# are more strongly associated with deterioration in this cohort.

secondary_dx_frequency = (
    secondary_dx_raw
    .groupby("secondary_diagnosis_description")
    .agg(
        n_rows=("secondary_diagnosis_description", "size"),
        n_cases=("case_no_deid_clean", "nunique")
    )
    .reset_index()
    .sort_values(["n_cases", "n_rows"], ascending=False)
)

# Merge diagnosis groups with case-level outcomes for positive-rate checking.
dx_case_outcomes = secondary_dx_features.merge(
    full_cohort[["case_no_deid_clean"] + label_cols].drop_duplicates("case_no_deid_clean"),
    on="case_no_deid_clean",
    how="left"
)

secondary_dx_group_summary = []
n_total_cases = dx_case_outcomes["case_no_deid_clean"].nunique()

for col in secondary_dx_group_cols:
    subset = dx_case_outcomes[dx_case_outcomes[col] == 1]
    n_group_cases = subset["case_no_deid_clean"].nunique()

    summary_row = {
        "secondary_dx_group": col,
        "n_cases": int(n_group_cases),
        "case_rate": n_group_cases / n_total_cases if n_total_cases > 0 else np.nan
    }

    for label_col in label_cols:
        summary_row[f"{label_col}_positive_cases"] = int(subset[label_col].sum()) if n_group_cases > 0 else 0
        summary_row[f"{label_col}_positive_case_rate"] = subset[label_col].mean() if n_group_cases > 0 else np.nan

    secondary_dx_group_summary.append(summary_row)

secondary_dx_group_summary = (
    pd.DataFrame(secondary_dx_group_summary)
    .sort_values("n_cases", ascending=False)
)

secondary_dx_features_path = f"{output_dir}/secondary_diagnosis_grouped_features.csv"
secondary_dx_group_summary_path = f"{output_dir}/secondary_diagnosis_group_summary_with_outcome_rates.csv"
secondary_dx_frequency_path = f"{output_dir}/secondary_diagnosis_frequency.csv"

secondary_dx_features.to_csv(secondary_dx_features_path, index=False)
secondary_dx_group_summary.to_csv(secondary_dx_group_summary_path, index=False)
secondary_dx_frequency.to_csv(secondary_dx_frequency_path, index=False)

print("\nSecondary diagnosis group summary with outcome rates:")
display(secondary_dx_group_summary)

print("Saved secondary diagnosis checking files to:")
print(secondary_dx_features_path)
print(secondary_dx_group_summary_path)
print(secondary_dx_frequency_path)


# ------------------------------------------------------------
# 5. Merge grouped diagnosis features into full cohort
# ------------------------------------------------------------
# Important notebook-running note:
#   In Colab, we often rerun one cell several times for debugging.
#   If this cell was already run once, full_cohort may already contain
#   secondary-diagnosis columns.
#
#   If we merge again without removing old versions, pandas will rename
#   duplicated columns as *_x and *_y. Then the clean column name
#   secondary_diagnosis_count will no longer exist, causing:
#       KeyError: 'secondary_diagnosis_count'
#
#   To make this cell safe to rerun, we first remove any old secondary
#   diagnosis columns from full_cohort, then merge the fresh version.

old_secondary_dx_cols = []

for col in secondary_dx_fill_cols:
    old_secondary_dx_cols.extend([
        col,
        f"{col}_x",
        f"{col}_y"
    ])

old_secondary_dx_cols = [
    col for col in old_secondary_dx_cols
    if col in full_cohort.columns
]

if len(old_secondary_dx_cols) > 0:
    print("Removing old secondary diagnosis columns before re-merging:")
    print(old_secondary_dx_cols)
    full_cohort = full_cohort.drop(columns=old_secondary_dx_cols)

# Make sure the merge key has the same clean format on both sides.
full_cohort["case_no_deid_clean"] = (
    full_cohort["case_no_deid_clean"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

secondary_dx_features["case_no_deid_clean"] = (
    secondary_dx_features["case_no_deid_clean"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

full_cohort = full_cohort.merge(
    secondary_dx_features,
    on="case_no_deid_clean",
    how="left",
    validate="one_to_one"
)

for col in secondary_dx_fill_cols:
    if col not in full_cohort.columns:
        raise KeyError(
            f"Expected diagnosis feature column was not created after merge: {col}. "
            "Please check secondary_dx_features and the merge key case_no_deid_clean."
        )
    full_cohort[col] = full_cohort[col].fillna(0).astype(int)

print("\nFull cohort after adding secondary diagnosis groups:")
print(full_cohort.shape)
print("Secondary diagnosis feature columns added:")
print(secondary_dx_fill_cols)


Secondary diagnosis source:
/content/drive/My Drive/merged_240920_240822_prepared/diagnosis_summary_all_240920_240822.csv

Cleaned secondary diagnosis rows: 8277
Unique cases with secondary diagnosis: 984

Secondary diagnosis grouped feature table:
(441, 22)


,case_no_deid_clean,secondary_diagnosis_count,secondary_dx_infection_sepsis,secondary_dx_pneumonia,secondary_dx_respiratory,secondary_dx_cardiac,secondary_dx_thrombosis_embolism,secondary_dx_renal_any,secondary_dx_acute_kidney_failure,secondary_dx_ckd_stage_3_5_or_dialysis,...,secondary_dx_electrolyte_acid_base_volume,secondary_dx_anemia_bleeding_coagulation,secondary_dx_malignancy,secondary_dx_metastatic_cancer,secondary_dx_neuro_delirium_stroke,secondary_dx_liver_cirrhosis_failure,secondary_dx_gi_acute,secondary_dx_frailty_pressure_ulcer_dysphagia_fall,secondary_dx_hypotension_shock,secondary_dx_severe_group_count
0,36f9ecbbb2,25,0,0,1,1,0,1,1,0,...,0,1,0,0,0,1,0,1,1,10
1,df9d9f5c32,6,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,4
2,690a82c936,11,1,0,0,0,0,1,0,1,...,0,1,0,0,0,0,0,0,0,6
3,69662b4977,6,1,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,3
4,12f6db609d,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1



Secondary diagnosis group summary with outcome rates:


,secondary_dx_group,n_cases,case_rate,label_hd_icu_positive_cases,label_hd_icu_positive_case_rate,label_o2_increase_positive_cases,label_o2_increase_positive_case_rate,label_iv_increase_positive_cases,label_iv_increase_positive_case_rate,label_met_positive_cases,label_met_positive_case_rate,label_death_positive_cases,label_death_positive_case_rate,label_any_deterioration_positive_cases,label_any_deterioration_positive_case_rate
17,secondary_dx_frailty_pressure_ulcer_dysphagia_...,230,0.521542,48,0.208696,18,0.078261,44,0.191304,4,0.017391,69,0.300000,126,0.547826
9,secondary_dx_hypertension,220,0.498866,39,0.177273,13,0.059091,40,0.181818,3,0.013636,53,0.240909,107,0.486364
8,secondary_dx_diabetes_complication,198,0.448980,29,0.146465,8,0.040404,30,0.151515,1,0.005051,37,0.186869,83,0.419192
5,secondary_dx_renal_any,157,0.356009,30,0.191083,12,0.076433,35,0.222930,3,0.019108,44,0.280255,87,0.554140
10,secondary_dx_electrolyte_acid_base_volume,132,0.299320,25,0.189394,10,0.075758,37,0.280303,2,0.015152,46,0.348485,83,0.628788
11,secondary_dx_anemia_bleeding_coagulation,119,0.269841,29,0.243697,10,0.084034,23,0.193277,2,0.016807,37,0.310924,72,0.605042
0,secondary_dx_infection_sepsis,106,0.240363,29,0.273585,12,0.113208,32,0.301887,2,0.018868,30,0.283019,67,0.632075
7,secondary_dx_ckd_stage_3_5_or_dialysis,101,0.229025,19,0.188119,5,0.049505,19,0.188119,1,0.009901,29,0.287129,52,0.514851
3,secondary_dx_cardiac,76,0.172336,23,0.302632,9,0.118421,20,0.263158,2,0.026316,31,0.407895,56,0.736842
6,secondary_dx_acute_kidney_failure,72,0.163265,16,0.222222,8,0.111111,23,0.319444,3,0.041667,21,0.291667,48,0.666667


Saved secondary diagnosis checking files to:
/content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/secondary_diagnosis_grouped_features.csv
/content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/secondary_diagnosis_group_summary_with_outcome_rates.csv
/content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/secondary_diagnosis_frequency.csv
Removing old secondary diagnosis columns before re-merging:
['secondary_diagnosis_count']

Full cohort after adding secondary diagnosis groups:
(441, 99)
Secondary diagnosis feature columns added:
['secondary_diagnosis_count', 'secondary_dx_severe_group_count', 'secondary_dx_infection_sepsis', 'secondary_dx_pneumonia', 'secondary_dx_respiratory', 'secondary_dx_cardiac', 'secondary_dx_thrombosis_embolism', 'secondary_dx_renal_any', 'secondary_dx_acute_kidney_failure', 'secondary_dx_ckd_stage_3_5_or_dialysis', 'secondary_dx_diabetes_comp

## Cell 8B explanation — case-level features copied into windows

Every generated window row needs patient-level features such as age, gender, admission type, data batch, and secondary diagnosis groups.

These features do not change from window to window for the same case, but they are copied into each row so each row is model-ready.


## Cell 8B: Prepare case-level patient and secondary-diagnosis features

### Non-technical note for Cell 8 — case-level features
This cell prepares the case-level variables that will be copied into every sliding window. These include patient features, event times used internally, and secondary-diagnosis group flags.

Some columns in this table are used only internally for label creation. Later, the notebook removes event-time columns before saving model-ready window datasets.

In [21]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# ============================================================
# Cell 8B: Prepare case-level patient and secondary-diagnosis features
# ============================================================
# These case-level features are copied into every window generated from
# the same patient/case.
#
# Important leakage rule:
#   Primary diagnosis features are NOT included because primary diagnosis
#   may be assigned after W36 admission/discharge.
#
# Diagnosis features included here are secondary-diagnosis groups only.
# ============================================================

patient_feature_cols = [
    "case_no_deid_clean",

    # Basic patient-level features available before prediction.
    "Age",
    "Gender",
    "Race",
    "Admit Type Description",
    "data_batch",
    "data_split",

    # Case-level outcome flags and event times are kept only inside
    # case_features so we can create window labels.
    # They are NOT written as model features in the final window datasets.
    "label_hd_icu",
    "label_o2_increase",
    "label_iv_increase",
    "label_met",
    "label_death",
    "label_any_deterioration",

    "event_time_hd_icu",
    "event_time_o2",
    "event_time_iv",
    "event_time_met",
    "event_time_death",
    "first_event_time",

    # Start/end of the monitoring episode.
    # Movement End is used only to know where retrospective windows can be built.
    # It is not written as a prediction feature.
    "W36 Start DT_dt",
    "Movement End DT_dt"
]

# Secondary-diagnosis features created in Cell 8A.
# These replace the old mixed primary/secondary diagnosis features.
diagnosis_feature_cols = [
    "secondary_diagnosis_count",
    "secondary_dx_severe_group_count"
] + secondary_dx_group_cols

case_feature_cols = patient_feature_cols + diagnosis_feature_cols

missing_case_feature_cols = [
    col for col in case_feature_cols
    if col not in full_cohort.columns
]

if len(missing_case_feature_cols) > 0:
    raise KeyError(
        "The following required case-level columns are missing: "
        f"{missing_case_feature_cols}"
    )

case_features = full_cohort[case_feature_cols].copy()

print("Prepared case-level feature table:")
print(case_features.shape)

print("\nPatient-level columns:")
print(patient_feature_cols)

print("\nSecondary diagnosis feature columns:")
print(diagnosis_feature_cols)

print("\nPreview:")
display(case_features.head())


Prepared case-level feature table:
(441, 42)

Patient-level columns:
['case_no_deid_clean', 'Age', 'Gender', 'Race', 'Admit Type Description', 'data_batch', 'data_split', 'label_hd_icu', 'label_o2_increase', 'label_iv_increase', 'label_met', 'label_death', 'label_any_deterioration', 'event_time_hd_icu', 'event_time_o2', 'event_time_iv', 'event_time_met', 'event_time_death', 'first_event_time', 'W36 Start DT_dt', 'Movement End DT_dt']

Secondary diagnosis feature columns:
['secondary_diagnosis_count', 'secondary_dx_severe_group_count', 'secondary_dx_infection_sepsis', 'secondary_dx_pneumonia', 'secondary_dx_respiratory', 'secondary_dx_cardiac', 'secondary_dx_thrombosis_embolism', 'secondary_dx_renal_any', 'secondary_dx_acute_kidney_failure', 'secondary_dx_ckd_stage_3_5_or_dialysis', 'secondary_dx_diabetes_complication', 'secondary_dx_hypertension', 'secondary_dx_electrolyte_acid_base_volume', 'secondary_dx_anemia_bleeding_coagulation', 'secondary_dx_malignancy', 'secondary_dx_metastatic

,case_no_deid_clean,Age,Gender,Race,Admit Type Description,data_batch,data_split,label_hd_icu,label_o2_increase,label_iv_increase,...,secondary_dx_hypertension,secondary_dx_electrolyte_acid_base_volume,secondary_dx_anemia_bleeding_coagulation,secondary_dx_malignancy,secondary_dx_metastatic_cancer,secondary_dx_neuro_delirium_stroke,secondary_dx_liver_cirrhosis_failure,secondary_dx_gi_acute,secondary_dx_frailty_pressure_ulcer_dysphagia_fall,secondary_dx_hypotension_shock
0,36f9ecbbb2,77.0,FEMALE,Chinese,Urgent,240920,train,0,0,0,...,1,0,1,0,0,0,1,0,1,1
1,df9d9f5c32,76.0,FEMALE,Chinese,Urgent,240920,train,0,0,0,...,1,0,1,1,0,0,0,0,0,0
2,690a82c936,49.0,FEMALE,Chinese,Urgent,240920,test,0,0,0,...,1,0,1,0,0,0,0,0,0,0
3,69662b4977,49.0,FEMALE,Chinese,Urgent,240920,test,0,0,1,...,0,0,1,0,0,0,0,0,1,0
4,12f6db609d,71.0,FEMALE,Chinese,Elective,240920,train,0,0,0,...,0,0,0,0,0,0,1,0,0,0


## Cell 9 explanation — charted vital signs

Charted vital signs are manually or clinically recorded measurements.

This cell reshapes them into a long format:

`case ID + vital time + vital name + vital value`

Long format makes it easier to summarize any vital sign inside any time window.


## Cell 9: Prepare charted vital signs for window aggregation

### Non-technical note for Cell 9 — charted vital signs
This cell converts charted vital signs into long format. Long format means every row is one vital-sign measurement, with columns for case ID, measurement time, vital name, and value.

Long format is easier for window-based feature extraction because the code can select all measurements inside a time window and summarize them.

In [23]:
# ------------------------------------------------------------
# Cell 9: Prepare charted vital signs in long format
# ------------------------------------------------------------
# Reader note:
# This cell prepares the manually charted vital-sign records.
#
# Why this step is needed:
#   Later feature engineering needs all vital signs in the same structure:
#
#       case ID | time | vital name | vital value
#
#   Some prepared datasets already store charted vitals in this long format.
#   Other datasets store vitals in wide format, such as HR, RR, SpO2 columns.
#
# This cell handles both situations safely.
# ------------------------------------------------------------

# ------------------------------------------------------------
# 1. Check that required upstream tables exist
# ------------------------------------------------------------

required_objects = ["charted_vitals", "full_cohort"]

for obj_name in required_objects:
    if obj_name not in globals():
        raise NameError(
            f"{obj_name} is not available. "
            "Please run the earlier data-loading and full-cohort cells first."
        )

print("Available charted_vitals columns:")
print(list(charted_vitals.columns))


# ------------------------------------------------------------
# 2. Keep only charted vital rows from selected full-source cases
# ------------------------------------------------------------

valid_case_ids = set(full_cohort["case_no_deid_clean"])

charted_clean = charted_vitals[
    charted_vitals["case_no_deid_clean"].isin(valid_case_ids)
].copy()

print("Charted records after keeping full-cohort cases:")
print(charted_clean.shape)


# ------------------------------------------------------------
# 3. Check the time column
# ------------------------------------------------------------
# Most previous notebook versions used created_datetime_dt as the charted
# vital-sign time column. If a future version uses another name, this block
# will print available columns so we can adjust it clearly.
# ------------------------------------------------------------

if "created_datetime_dt" not in charted_clean.columns:
    raise KeyError(
        "created_datetime_dt is not found in charted_vitals. "
        "Please check the printed charted_vitals columns above and identify "
        "the charted vital-sign time column."
    )

charted_clean["created_datetime_dt"] = pd.to_datetime(
    charted_clean["created_datetime_dt"],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. Standardize vital-sign names
# ------------------------------------------------------------

def standardize_vital_name(vital_name):
    """
    Convert source-specific vital names into one clinical vital-sign name.

    Why:
        In this dataset, charted and sensor values are treated as
        observations of the same underlying clinical variable when their
        meaning is comparable.

    Examples:
        charted_HR                  -> HR
        charted_charted_HR          -> HR
        sensor_RR                   -> RR
        body_temperature            -> temperature
        sensor_body_temperature     -> temperature

    Important:
        skin_temperature is intentionally kept separate.
        Skin temperature is not the same clinical variable as body temperature.
    """

    if pd.isna(vital_name):
        return np.nan

    name = str(vital_name).strip()

    # Remove repeated source prefixes until none remain.
    # This fixes cases like charted_charted_HR or sensor_sensor_RR.
    changed = True
    while changed:
        changed = False
        for prefix in ["charted_", "sensor_"]:
            if name.startswith(prefix):
                name = name[len(prefix):]
                changed = True

    name_lower = name.lower().replace("_", " ").strip()

    vital_name_map = {
        "hr": "HR",
        "heart rate": "HR",
        "heart rate value": "HR",

        "rr": "RR",
        "respiratory rate": "RR",

        "spo2": "SpO2",
        "spo₂": "SpO2",
        "oxygen saturation": "SpO2",

        "temperature": "temperature",
        "body temperature": "temperature",

        # Keep this separate from clinical/body temperature.
        "skin temperature": "skin_temperature",

        "sbp": "SBP",
        "systolic bp": "SBP",
        "systolic blood pressure": "SBP",

        "dbp": "DBP",
        "diastolic bp": "DBP",
        "diastolic blood pressure": "DBP"
    }

    return vital_name_map.get(name_lower, name)


expected_charted_vitals = [
    "HR",
    "RR",
    "SpO2",
    "temperature",
    "SBP",
    "DBP"
]


# ------------------------------------------------------------
# 5. Detect whether charted vitals are already in long format
# ------------------------------------------------------------
# Long format means:
#   one row = one vital-sign measurement
#
# It usually has:
#   - one column storing the vital-sign name
#   - one column storing the value, such as value_numeric
# ------------------------------------------------------------

possible_vital_name_cols = [
    "vital_name",
    "vital sign",
    "vital_sign",
    "vital_sign_name",
    "vital_type",
    "parameter",
    "parameter_name",
    "variable",
    "variable_name",
    "item",
    "item_name",
    "measurement",
    "measurement_name",
    "observation",
    "observation_name",
    "description",
    "name"
]

existing_vital_name_cols = [
    col for col in possible_vital_name_cols
    if col in charted_clean.columns
]

# Broader search for unusual but plausible column names.
for col in charted_clean.columns:
    col_lower = str(col).lower()

    if (
        ("vital" in col_lower or
         "parameter" in col_lower or
         "description" in col_lower or
         "measurement" in col_lower or
         "item" in col_lower)
        and col not in existing_vital_name_cols
        and col not in ["case_no_deid_clean", "created_datetime_dt"]
    ):
        existing_vital_name_cols.append(col)


# ------------------------------------------------------------
# 6A. Case A: charted_clean is already long format
# ------------------------------------------------------------

if ("value_numeric" in charted_clean.columns) and (len(existing_vital_name_cols) > 0):

    vital_name_col = existing_vital_name_cols[0]

    print("Detected charted vitals as LONG format.")
    print("Vital-name column used:", vital_name_col)
    print("Value column used: value_numeric")

    charted_long = charted_clean[
        ["case_no_deid_clean", "created_datetime_dt", vital_name_col, "value_numeric"]
    ].copy()

    charted_long = charted_long.rename(columns={
        "created_datetime_dt": "vital_time",
        vital_name_col: "vital_name",
        "value_numeric": "vital_value"
    })

    charted_long["vital_name"] = charted_long["vital_name"].apply(
        standardize_vital_name
    )

    charted_long = charted_long[
        charted_long["vital_name"].isin(expected_charted_vitals)
    ].copy()


# ------------------------------------------------------------
# 6B. Case B: charted_clean is wide format
# ------------------------------------------------------------

else:

    print("Could not confirm long format. Trying WIDE-format detection.")

    charted_vital_candidate_map = {
        "HR": ["charted_HR", "HR", "heart_rate", "Heart Rate"],
        "RR": ["charted_RR", "RR", "respiratory_rate", "Respiratory Rate"],
        "SpO2": ["charted_SpO2", "SpO2", "spo2", "oxygen_saturation", "Oxygen Saturation"],
        "temperature": ["charted_temperature", "temperature", "body_temperature", "Temperature"],
        "SBP": ["charted_SBP", "SBP", "systolic_bp", "Systolic BP", "Systolic Blood Pressure"],
        "DBP": ["charted_DBP", "DBP", "diastolic_bp", "Diastolic BP", "Diastolic Blood Pressure"]
    }

    charted_value_cols = []
    charted_column_to_standard_name = {}

    for standard_name, candidate_cols in charted_vital_candidate_map.items():

        matched_col = None

        for candidate_col in candidate_cols:
            if candidate_col in charted_clean.columns:
                matched_col = candidate_col
                break

        if matched_col is not None:
            charted_value_cols.append(matched_col)
            charted_column_to_standard_name[matched_col] = standard_name
        else:
            print(f"Warning: no charted column found for {standard_name}")

    if len(charted_value_cols) == 0:
        print("Available charted_clean columns:")
        print(list(charted_clean.columns))

        raise ValueError(
            "No usable charted vital-sign columns were found. "
            "The table is neither recognized as long format nor wide format. "
            "Please inspect charted_clean.columns and identify the column "
            "that stores vital-sign names."
        )

    print("Detected charted vitals as WIDE format.")
    print("Charted vital columns detected and used:")

    for original_col, standard_name in charted_column_to_standard_name.items():
        print(f"  {original_col}  ->  {standard_name}")

    charted_long = charted_clean.melt(
        id_vars=["case_no_deid_clean", "created_datetime_dt"],
        value_vars=charted_value_cols,
        var_name="vital_name",
        value_name="vital_value"
    )

    charted_long = charted_long.rename(columns={
        "created_datetime_dt": "vital_time"
    })

    charted_long["vital_name"] = charted_long["vital_name"].map(
        charted_column_to_standard_name
    )


# ------------------------------------------------------------
# 7. Final cleaning after either long-format or wide-format conversion
# ------------------------------------------------------------

charted_long = charted_long[
    charted_long["vital_value"].notna()
].copy()

charted_long["vital_time"] = pd.to_datetime(
    charted_long["vital_time"],
    errors="coerce"
)

charted_long["vital_value"] = pd.to_numeric(
    charted_long["vital_value"],
    errors="coerce"
)

charted_long = charted_long[
    charted_long["vital_time"].notna() &
    charted_long["vital_value"].notna()
].copy()

charted_long["vital_name"] = charted_long["vital_name"].apply(
    standardize_vital_name
)

print("Charted vital table prepared:")
print(charted_long.shape)

print("\nStandardized charted vital names:")
print(sorted(charted_long["vital_name"].dropna().unique()))

print("\nPreview:")
display(charted_long.head())

Available charted_vitals columns:
['case_no_deid_clean', 'created_datetime_dt', 'vital_type', 'vital_label_raw', 'Value Text', 'value_numeric', 'UOM', 'is_linked_to_case_master', 'data_batch']
Charted records after keeping full-cohort cases:
(209900, 9)
Detected charted vitals as LONG format.
Vital-name column used: vital_type
Value column used: value_numeric
Charted vital table prepared:
(208601, 4)

Standardized charted vital names:
['DBP', 'HR', 'RR', 'SBP', 'SpO2', 'temperature']

Preview:


,case_no_deid_clean,vital_time,vital_name,vital_value
824,08e0e56a6c,2024-08-26 18:57:57,DBP,77.0
825,08e0e56a6c,2024-08-26 18:57:57,SBP,117.0
826,08e0e56a6c,2024-08-26 18:57:57,HR,66.0
827,08e0e56a6c,2024-08-26 18:57:57,SpO2,97.0
828,08e0e56a6c,2024-08-26 18:57:57,RR,19.0


## Cell 10: Prepare Respiree sensor vital signs for window aggregation

### Non-technical note for Cell 10 — sensor vital signs
This cell prepares Respiree sensor vital signs in the same long format as charted vital signs. Having both sources in the same structure makes it possible to merge them into unified clinical variables.

The project decision is to treat charted and sensor values as measurements of the same underlying vital sign when they refer to the same clinical variable.

In [24]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

sensor_clean = sensor_timeseries[
    sensor_timeseries["case_no_deid_clean"].isin(valid_case_ids)
].copy()

# Keep only linked sensor records inside the patient episode.
sensor_clean = sensor_clean[
    (sensor_clean["is_linked_to_case"] == 1) &
    (sensor_clean["within_patientinfo_episode"] == 1) &
    (sensor_clean["sensor_datetime_dt"].notna())
].copy()

# Sensor vital columns available in the Respiree time-series table.
sensor_vital_cols = [
    "RR",
    "HR",
    "SpO2",
    "skin_temperature",
    "body_temperature"
]

# Convert wide sensor format into long format.
sensor_long = sensor_clean.melt(
    id_vars=["case_no_deid_clean", "sensor_datetime_dt"],
    value_vars=sensor_vital_cols,
    var_name="vital_name",
    value_name="vital_value"
)

# Keep non-missing values only.
sensor_long = sensor_long[
    sensor_long["vital_value"].notna()
].copy()

sensor_long = sensor_long.rename(columns={
    "sensor_datetime_dt": "vital_time"
})

# V4 design:
# Do NOT add a sensor_ prefix.
# HR, RR, SpO2, and body_temperature are merged with charted records.
# skin_temperature remains a separate variable.
sensor_long["vital_name"] = sensor_long["vital_name"].apply(standardize_vital_name)

print("Sensor vital table prepared:")
print(sensor_long.shape)
print("\nStandardized sensor vital names:")
print(sorted(sensor_long["vital_name"].dropna().unique()))
print("\nPreview:")
display(sensor_long.head())


Sensor vital table prepared:
(753130, 4)

Standardized sensor vital names:
['HR', 'RR', 'SpO2', 'skin_temperature', 'temperature']

Preview:


,case_no_deid_clean,vital_time,vital_name,vital_value
0,08e0e56a6c,2024-08-28 07:13:54,RR,25.0
1,08e0e56a6c,2024-08-28 07:16:41,RR,25.0
2,08e0e56a6c,2024-08-28 07:19:28,RR,27.0
3,08e0e56a6c,2024-08-28 07:22:14,RR,24.0
4,08e0e56a6c,2024-08-28 07:24:57,RR,25.0


## Cell 11: Combine charted vitals and sensor vitals

### Non-technical note for Cell 11 — unified vital stream
This cell combines charted and sensor vital-sign records into one table.

The key design choice is: **one clinical vital sign should have one feature stream**. For example, charted HR and sensor HR are both renamed to `HR`. The model will then see `HR_mean`, `HR_min`, `HR_max`, etc., rather than separate charted/sensor HR features.

`skin_temperature` remains separate because skin temperature is not the same clinical variable as body temperature.

In [25]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

all_vitals_long = pd.concat(
    [charted_long, sensor_long],
    axis=0,
    ignore_index=True
)

# Remove impossible missing values again.
all_vitals_long = all_vitals_long[
    all_vitals_long["vital_time"].notna() &
    all_vitals_long["vital_value"].notna() &
    all_vitals_long["vital_name"].notna()
].copy()

all_vitals_long["vital_time"] = pd.to_datetime(
    all_vitals_long["vital_time"],
    errors="coerce"
)

all_vitals_long = all_vitals_long.dropna(subset=["vital_time"])

# Sort once and build a dictionary by case.
# This avoids repeatedly scanning the full vital table inside every loop.
all_vitals_long = all_vitals_long.sort_values([
    "case_no_deid_clean", "vital_time"
]).reset_index(drop=True)

vitals_by_case = {
    case_id: group.copy()
    for case_id, group in all_vitals_long.groupby("case_no_deid_clean", sort=False)
}

empty_vital_table = all_vitals_long.iloc[0:0].copy()

print("Combined unified vital table:")
print(all_vitals_long.shape)

print("\nUnified vital names:")
print(sorted(all_vitals_long["vital_name"].unique()))

print("\nNumber of cases with at least one vital record:")
print(len(vitals_by_case))

print("\nImportant note:")
print("HR, RR, SpO2, and temperature now combine charted and sensor records.")
print("skin_temperature remains separate from temperature.")

Combined unified vital table:
(961731, 4)

Unified vital names:
['DBP', 'HR', 'RR', 'SBP', 'SpO2', 'skin_temperature', 'temperature']

Number of cases with at least one vital record:
441

Important note:
HR, RR, SpO2, and temperature now combine charted and sensor records.
skin_temperature remains separate from temperature.


## Cell 12: Helper functions for unified vital-sign feature extraction

### Non-technical note for Cell 12 — vital-sign feature functions
This cell defines reusable functions that convert raw vital-sign measurements into model features.

For each vital sign inside a window, the code creates summaries such as mean, min, max, standard deviation, count, first, last, slope, delta, time since last measurement, and missing flag. These summaries are designed to capture both the current level and the recent trend of physiology.

The code also creates baseline/recent comparison features and recent 60-minute / 30-minute features. These help capture acute deterioration that might be hidden by a long-window average.

In [26]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# ============================================================
# Cell 12: Helper functions for unified vital-sign feature extraction
# ============================================================
# The vital table is already unified:
#   charted HR + sensor HR -> HR
#   charted RR + sensor RR -> RR
#   charted SpO2 + sensor SpO2 -> SpO2
#   charted temperature + sensor body_temperature -> temperature
#
# skin_temperature remains separate because it is not the same clinical
# variable as body temperature.
# ============================================================

all_vital_names = sorted(all_vitals_long["vital_name"].dropna().unique())

print("Number of unified vital types:")
print(len(all_vital_names))

print("\nUnified vital types:")
print(all_vital_names)


def summarize_vitals_in_window(vital_subset, window_end, vital_names, prefix=""):
    """
    Summarize vital signs inside one selected time window.

    No future data is used. The input vital_subset should already be
    restricted to the period before window_end.

    Features for each vital sign:
        mean, min, max, std, count, last, first,
        delta, slope, time_since_last, missing_flag.
    """

    feature_dict = {}

    for vital_name in vital_names:

        base = f"{prefix}{vital_name}"
        group = vital_subset[vital_subset["vital_name"] == vital_name].copy()

        if group.empty:
            feature_dict[f"{base}_mean"] = np.nan
            feature_dict[f"{base}_min"] = np.nan
            feature_dict[f"{base}_max"] = np.nan
            feature_dict[f"{base}_std"] = np.nan
            feature_dict[f"{base}_count"] = 0
            feature_dict[f"{base}_last"] = np.nan
            feature_dict[f"{base}_first"] = np.nan
            feature_dict[f"{base}_delta"] = np.nan
            feature_dict[f"{base}_slope"] = np.nan
            feature_dict[f"{base}_time_since_last"] = np.nan
            feature_dict[f"{base}_missing_flag"] = 1
            continue

        group = group.sort_values("vital_time")
        values = group["vital_value"]

        first_value = values.iloc[0]
        last_value = values.iloc[-1]
        delta_value = last_value - first_value

        first_time = group["vital_time"].iloc[0]
        last_time = group["vital_time"].iloc[-1]

        time_gap_hours = (last_time - first_time).total_seconds() / 3600
        time_since_last_hours = (window_end - last_time).total_seconds() / 3600

        if len(group) >= 2 and time_gap_hours > 0:
            slope_value = delta_value / time_gap_hours
        else:
            slope_value = np.nan

        feature_dict[f"{base}_mean"] = values.mean()
        feature_dict[f"{base}_min"] = values.min()
        feature_dict[f"{base}_max"] = values.max()
        feature_dict[f"{base}_std"] = 0 if values.count() == 1 else values.std()
        feature_dict[f"{base}_count"] = values.count()
        feature_dict[f"{base}_last"] = last_value
        feature_dict[f"{base}_first"] = first_value
        feature_dict[f"{base}_delta"] = delta_value
        feature_dict[f"{base}_slope"] = slope_value
        feature_dict[f"{base}_time_since_last"] = time_since_last_hours
        feature_dict[f"{base}_missing_flag"] = 0

    return feature_dict


def summarize_short_recent_features(vital_subset, vital_names, prefix):
    """
    Summarize short recent vital-sign information.

    This is used for:
        last60min_<vital>_...
        last30min_<vital>_...

    Features are intentionally concise:
        mean, min, max, std, count, last.
    """

    feature_dict = {}

    for vital_name in vital_names:

        base = f"{prefix}_{vital_name}"
        group = vital_subset[vital_subset["vital_name"] == vital_name].copy()

        if group.empty:
            feature_dict[f"{base}_mean"] = np.nan
            feature_dict[f"{base}_min"] = np.nan
            feature_dict[f"{base}_max"] = np.nan
            feature_dict[f"{base}_std"] = np.nan
            feature_dict[f"{base}_count"] = 0
            feature_dict[f"{base}_last"] = np.nan
            continue

        group = group.sort_values("vital_time")
        values = group["vital_value"]

        feature_dict[f"{base}_mean"] = values.mean()
        feature_dict[f"{base}_min"] = values.min()
        feature_dict[f"{base}_max"] = values.max()
        feature_dict[f"{base}_std"] = 0 if values.count() == 1 else values.std()
        feature_dict[f"{base}_count"] = values.count()
        feature_dict[f"{base}_last"] = values.iloc[-1]

    return feature_dict


def add_vital_availability_features(feature_dict, vital_names):
    """
    Add simple vital-availability features for one window.

    Why this may help:
        Missingness itself can be informative. A window with all core
        vitals available is different from a window with only one signal.

    These features do not use future information. They are calculated
    from the count features already created inside the observation window.
    """

    core_vitals = ["HR", "RR", "SpO2", "temperature", "SBP", "DBP"]

    available_vitals = [
        vital for vital in vital_names
        if feature_dict.get(f"{vital}_count", 0) > 0
    ]

    available_core_vitals = [
        vital for vital in core_vitals
        if feature_dict.get(f"{vital}_count", 0) > 0
    ]

    n_all = len(vital_names)
    n_core = len(core_vitals)

    return {
        "available_vital_type_count": len(available_vitals),
        "missing_vital_type_count": n_all - len(available_vitals),
        "core_available_vital_count": len(available_core_vitals),
        "core_missing_vital_count": n_core - len(available_core_vitals),
        "all_core_vitals_available_flag": int(len(available_core_vitals) == n_core)
    }



def add_recent_baseline_contrast_features(feature_dict, vital_names):
    """
    Add recent-minus-baseline contrast features.

    The observation window is split into two halves:
        baseline = earlier half
        recent   = later half

    These features describe whether the patient's recent status is worse
    than their own earlier status.
    """

    # Keep the contrast set concise and interpretable.
    # Delta contrast is excluded because it is harder to explain and can
    # duplicate other trend features.
    contrast_metrics = ["mean", "min", "max", "count"]

    for vital_name in vital_names:
        for metric in contrast_metrics:

            recent_col = f"recent_{vital_name}_{metric}"
            baseline_col = f"baseline_{vital_name}_{metric}"
            contrast_col = f"recent_minus_baseline_{vital_name}_{metric}"

            recent_value = feature_dict.get(recent_col, np.nan)
            baseline_value = feature_dict.get(baseline_col, np.nan)

            feature_dict[contrast_col] = recent_value - baseline_value

    return feature_dict


Number of unified vital types:
7

Unified vital types:
['DBP', 'HR', 'RR', 'SBP', 'SpO2', 'skin_temperature', 'temperature']


## Cell 13: Build reusable window features, then add cascade labels

### Non-technical note for Cell 13 — feature base and cascade history
This cell creates the expensive part of the dataset: one row per case-window with all features.

In v8, the notebook does **not** discard windows after the first event. Instead, it keeps later windows and adds prior-event features. This allows cascade modelling.

For example, if O2 increase already happened before the current window end, the row will have `prior_o2 = 1` and `time_since_prior_o2_hours` showing how long ago it happened. These are valid predictors because they are known before prediction time.

In [27]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# ============================================================
# Cell 13: Build reusable window features, then add cascade labels
# ============================================================
# Runtime improvement:
#   Vital-sign features depend on the observation window length m,
#   but they do NOT depend on the prediction horizon h.
#
# Cascade modelling update:
#   Earlier versions discarded all windows after the first deterioration.
#   This version keeps later windows, because a prior event can be a real,
#   known clinical history at prediction time.
#
#   Example:
#       O2 increase already happened before window_end.
#       It is valid to use prior_o2 = 1 to predict later MET / HD-ICU.
#
# Leakage control:
#   1. Only vital records before window_end are used.
#   2. Prior-event features use only events before window_end.
#   3. A target is eligible only if that same event has not already happened.
#   4. Internal event timestamps are removed before final CSV saving.
# ============================================================

target_cols = [
    "y_any",
    "y_hd_icu",
    "y_o2",
    "y_iv",
    "y_met",
    "y_death"
]

eligibility_cols = [
    "eligible_y_any",
    "eligible_y_hd_icu",
    "eligible_y_o2",
    "eligible_y_iv",
    "eligible_y_met",
    "eligible_y_death"
]

# Mapping used consistently for labels, eligibility, and prior-event features.
event_map = {
    "hd_icu": "__event_time_hd_icu",
    "o2": "__event_time_o2",
    "iv": "__event_time_iv",
    "met": "__event_time_met",
    "death": "__event_time_death"
}

internal_time_cols = ["__movement_end_dt"] + list(event_map.values())


def is_event_in_horizon(event_time, window_end, horizon_end):
    """
    Return 1 if an event happens inside the prediction horizon.

    Observation window: [window_start, window_end)
    Prediction horizon: [window_end, horizon_end)
    """

    if pd.isna(event_time):
        return 0

    return int((event_time >= window_end) and (event_time < horizon_end))


def add_prior_event_features(row_dict, case_row, window_end):
    """
    Add event-history features known at prediction time.

    These are valid cascade-model features because they only use events
    that happened before window_end.

    Created examples:
        prior_o2 = 1 if O2 increase happened before window_end
        time_since_prior_o2_hours = hours since that O2 increase
        prior_event_count = number of prior deterioration event types
    """

    prior_flags = []
    prior_times = []

    for event_name, event_col in {
        "hd_icu": "event_time_hd_icu",
        "o2": "event_time_o2",
        "iv": "event_time_iv",
        "met": "event_time_met",
        "death": "event_time_death"
    }.items():

        event_time = case_row[event_col]
        prior_flag = int(pd.notna(event_time) and event_time < window_end)

        row_dict[f"prior_{event_name}"] = prior_flag

        if prior_flag == 1:
            row_dict[f"time_since_prior_{event_name}_hours"] = (
                window_end - event_time
            ).total_seconds() / 3600
            prior_times.append(event_time)
        else:
            row_dict[f"time_since_prior_{event_name}_hours"] = np.nan

        prior_flags.append(prior_flag)

    row_dict["prior_event_count"] = int(sum(prior_flags))
    row_dict["any_prior_deterioration_flag"] = int(sum(prior_flags) > 0)

    if len(prior_times) > 0:
        first_prior_time = min(prior_times)
        most_recent_prior_time = max(prior_times)

        row_dict["time_since_first_prior_event_hours"] = (
            window_end - first_prior_time
        ).total_seconds() / 3600

        row_dict["time_since_most_recent_prior_event_hours"] = (
            window_end - most_recent_prior_time
        ).total_seconds() / 3600
    else:
        row_dict["time_since_first_prior_event_hours"] = np.nan
        row_dict["time_since_most_recent_prior_event_hours"] = np.nan

    return row_dict


def build_window_feature_base(case_table, vitals_by_case, m_hours, split_name):
    """
    Build feature rows for one observation-window length and one split.

    This function does NOT add prediction-horizon labels.
    It creates reusable feature rows that can later be labelled for
    h = 4, 6, and 12.

    Important cascade difference:
        We no longer discard windows after the first deterioration event.
        Instead, prior deterioration history is added as valid features.
    """

    selected_cases = case_table[case_table["data_split"] == split_name].copy()
    rows = []

    m_delta = pd.Timedelta(hours=m_hours)
    step_delta = pd.Timedelta(hours=1)
    last60_delta = pd.Timedelta(minutes=60)
    last30_delta = pd.Timedelta(minutes=30)

    for _, case_row in selected_cases.iterrows():

        case_id = case_row["case_no_deid_clean"]
        st = case_row["W36 Start DT_dt"]
        et = case_row["Movement End DT_dt"]

        if pd.isna(st) or pd.isna(et) or et <= st:
            continue

        case_vitals = vitals_by_case.get(case_id, empty_vital_table)
        window_start = st

        while window_start + m_delta <= et:

            window_end = window_start + m_delta

            # Vital records inside the observation window only.
            vital_subset = case_vitals[
                (case_vitals["vital_time"] >= window_start) &
                (case_vitals["vital_time"] < window_end)
            ]

            vital_features = summarize_vitals_in_window(
                vital_subset=vital_subset,
                window_end=window_end,
                vital_names=all_vital_names,
                prefix=""
            )

            # Baseline/recent split: first half vs second half.
            split_time = window_start + (m_delta / 2)

            baseline_subset = vital_subset[
                (vital_subset["vital_time"] >= window_start) &
                (vital_subset["vital_time"] < split_time)
            ]

            recent_subset = vital_subset[
                (vital_subset["vital_time"] >= split_time) &
                (vital_subset["vital_time"] < window_end)
            ]

            vital_features.update(
                summarize_vitals_in_window(
                    vital_subset=baseline_subset,
                    window_end=split_time,
                    vital_names=all_vital_names,
                    prefix="baseline_"
                )
            )

            vital_features.update(
                summarize_vitals_in_window(
                    vital_subset=recent_subset,
                    window_end=window_end,
                    vital_names=all_vital_names,
                    prefix="recent_"
                )
            )

            vital_features = add_recent_baseline_contrast_features(
                feature_dict=vital_features,
                vital_names=all_vital_names
            )

            # Fixed short recent windows.
            last60_subset = vital_subset[
                (vital_subset["vital_time"] >= window_end - last60_delta) &
                (vital_subset["vital_time"] < window_end)
            ]

            last30_subset = vital_subset[
                (vital_subset["vital_time"] >= window_end - last30_delta) &
                (vital_subset["vital_time"] < window_end)
            ]

            vital_features.update(
                summarize_short_recent_features(
                    vital_subset=last60_subset,
                    vital_names=all_vital_names,
                    prefix="last60min"
                )
            )

            vital_features.update(
                summarize_short_recent_features(
                    vital_subset=last30_subset,
                    vital_names=all_vital_names,
                    prefix="last30min"
                )
            )

            # Window-level vital-availability features.
            vital_features.update(
                add_vital_availability_features(
                    feature_dict=vital_features,
                    vital_names=all_vital_names
                )
            )

            row = {
                "case_no_deid_clean": case_id,
                "data_split": split_name,
                "m_hours": m_hours,
                "window_start": window_start,
                "window_end": window_end,

                # Internal columns for labelling/filtering only.
                # These will be removed before saving the final dataset.
                "__movement_end_dt": et,
                "__event_time_hd_icu": case_row["event_time_hd_icu"],
                "__event_time_o2": case_row["event_time_o2"],
                "__event_time_iv": case_row["event_time_iv"],
                "__event_time_met": case_row["event_time_met"],
                "__event_time_death": case_row["event_time_death"],

                # Patient-level time feature.
                "hours_since_w36_start": (
                    window_end - st
                ).total_seconds() / 3600,

                # Patient-level baseline features.
                "Age": case_row["Age"],
                "Gender": case_row["Gender"],
                "Race": case_row["Race"],
                "Admit Type Description": case_row["Admit Type Description"],
                "data_batch": case_row["data_batch"]
            }

            # Valid cascade features: event history before window_end.
            row = add_prior_event_features(
                row_dict=row,
                case_row=case_row,
                window_end=window_end
            )

            # Secondary diagnosis grouped features.
            for col in diagnosis_feature_cols:
                row[col] = case_row[col]

            # Vital-sign features.
            row.update(vital_features)

            rows.append(row)
            window_start += step_delta

    return pd.DataFrame(rows)


def add_horizon_labels(feature_base_df, h_hours):
    """
    Add cascade-style window-level labels for one prediction horizon.

    Full-horizon rule:
        Keep only windows where horizon_end <= Movement End DT.

    Cascade target rule for each event type:
        If the event already happened before window_end:
            eligible_y_event = 0
            y_event = NaN
        Else if the event happens inside [window_end, horizon_end):
            eligible_y_event = 1
            y_event = 1
        Else:
            eligible_y_event = 1
            y_event = 0

    This avoids leakage because events after window_end are never used as
    features. Previous events are allowed only as prior-event features.
    """

    h_delta = pd.Timedelta(hours=h_hours)

    if feature_base_df.empty:
        labelled = feature_base_df.copy()
        labelled["h_hours"] = h_hours
        labelled["horizon_end"] = pd.NaT
        for col in target_cols + eligibility_cols:
            labelled[col] = pd.Series(dtype="float")
        labelled = labelled.drop(columns=[col for col in internal_time_cols if col in labelled.columns])
        return labelled

    labelled = feature_base_df.copy()
    labelled["h_hours"] = h_hours
    labelled["horizon_end"] = labelled["window_end"] + h_delta

    # Keep only windows where the full prediction horizon is observable.
    labelled = labelled[
        labelled["horizon_end"] <= labelled["__movement_end_dt"]
    ].copy()

    if labelled.empty:
        return labelled.drop(columns=[col for col in internal_time_cols if col in labelled.columns])

    event_target_pairs = {
        "hd_icu": "y_hd_icu",
        "o2": "y_o2",
        "iv": "y_iv",
        "met": "y_met",
        "death": "y_death"
    }

    for event_name, target_col in event_target_pairs.items():

        event_col = event_map[event_name]
        eligible_col = f"eligible_{target_col}"

        already_happened = (
            labelled[event_col].notna() &
            (labelled[event_col] < labelled["window_end"])
        )

        labelled[eligible_col] = (~already_happened).astype(int)

        labelled[target_col] = np.where(
            already_happened,
            np.nan,
            [
                is_event_in_horizon(event_time, window_end, horizon_end)
                for event_time, window_end, horizon_end in zip(
                    labelled[event_col],
                    labelled["window_end"],
                    labelled["horizon_end"]
                )
            ]
        )

    # y_any means any NEW not-yet-occurred event within the horizon.
    labelled["eligible_y_any"] = (
        labelled[[
            "eligible_y_hd_icu",
            "eligible_y_o2",
            "eligible_y_iv",
            "eligible_y_met",
            "eligible_y_death"
        ]]
        .max(axis=1)
        .astype(int)
    )

    labelled["y_any"] = (
        labelled[["y_hd_icu", "y_o2", "y_iv", "y_met", "y_death"]]
        .max(axis=1, skipna=True)
    )

    labelled.loc[labelled["eligible_y_any"] == 0, "y_any"] = np.nan

    # Remove internal labelling columns before saving.
    labelled = labelled.drop(columns=[col for col in internal_time_cols if col in labelled.columns])

    # Put common metadata, labels, and eligibility flags first for readability.
    front_cols = [
        "case_no_deid_clean", "data_split", "m_hours", "h_hours",
        "window_start", "window_end", "horizon_end"
    ] + target_cols + eligibility_cols

    remaining_cols = [col for col in labelled.columns if col not in front_cols]
    labelled = labelled[front_cols + remaining_cols]

    return labelled


def summarize_horizon_censoring(feature_base_df, m_hours, h_hours, split_name):
    """
    Count how many reusable feature rows are removed by the full-horizon rule.

    Rule used in this notebook:
        Retain window only if horizon_end <= Movement End DT.

    Interpretation:
        Removed windows are late-stay windows where the full prediction
        horizon is not observable before Movement End DT.
    """

    n_before = len(feature_base_df)

    if n_before == 0:
        n_after = 0
    else:
        horizon_end = feature_base_df["window_end"] + pd.Timedelta(hours=h_hours)
        n_after = int((horizon_end <= feature_base_df["__movement_end_dt"]).sum())

    n_removed = n_before - n_after

    return {
        "m_hours": m_hours,
        "h_hours": h_hours,
        "split": split_name,
        "windows_before_horizon_filter": n_before,
        "windows_after_horizon_filter": n_after,
        "removed_windows": n_removed,
        "removed_rate": n_removed / n_before if n_before > 0 else np.nan,
        "censoring_rule": "Retain if horizon_end <= Movement End DT; removed windows fail this rule."
    }


## Cell 14: Optional example cascade dataset build: window length = 6h, prediction horizon = 12h

### Non-technical note for Cell 14 — optional example run
This cell is only for testing. It can build a small example dataset to check whether the functions work.

By default, `RUN_EXAMPLE_DATASET = False`, so this cell does not run expensive generation. Turn it on only when debugging.

In [28]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# ============================================================
# Cell 14: Optional example dataset build
# Example: m = 6h, h = 12h
# ============================================================
# This is useful for debugging, but it can take time.
# Keep RUN_EXAMPLE_DATASET = False in Cell 1 for the full production run.

# RUN_EXAMPLE_DATASET = True

example_m = 6
example_h = 12

if RUN_EXAMPLE_DATASET:

    example_feature_base = build_window_feature_base(
        case_table=case_features,
        vitals_by_case=vitals_by_case,
        m_hours=example_m,
        split_name="train"
    )

    train_example = add_horizon_labels(example_feature_base, h_hours=example_h)

    val_feature_base = build_window_feature_base(
        case_table=case_features,
        vitals_by_case=vitals_by_case,
        m_hours=example_m,
        split_name="validation"
    )

    val_example = add_horizon_labels(val_feature_base, h_hours=example_h)

    test_feature_base = build_window_feature_base(
        case_table=case_features,
        vitals_by_case=vitals_by_case,
        m_hours=example_m,
        split_name="test"
    )

    test_example = add_horizon_labels(test_feature_base, h_hours=example_h)

    print("Example dataset sizes:")
    print("Train:", train_example.shape)
    print("Validation:", val_example.shape)
    print("Test:", test_example.shape)

    print("\nPositive window rates by target:")
    print("Train:")
    display(train_example[target_cols].mean())

    print("Validation:")
    display(val_example[target_cols].mean())

    print("Test:")
    display(test_example[target_cols].mean())

else:
    train_example = None
    val_example = None
    test_example = None
    print("Skipped example dataset build because RUN_EXAMPLE_DATASET = False.")

Skipped example dataset build because RUN_EXAMPLE_DATASET = False.


## Cell 15: Save one example dataset

### Non-technical note for Cell 16–17 — full dataset generation
This is the main long-running cell. It builds feature bases for each observation window length and then adds labels for each prediction horizon.

The speed improvement is important: the expensive vital-sign features depend on the observation window length `m`, but not on the horizon `h`. Therefore, v8 builds the feature base once per `m`, then reuses it for `h = 4, 6, 12`.

The horizon rule is also important: windows are kept only when `horizon_end <= Movement End DT`. This means every saved window has full follow-up inside the W36 monitoring period.

In [29]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

if RUN_EXAMPLE_DATASET:
    example_folder = f"{output_dir}/m{example_m}_h{example_h}"
    os.makedirs(example_folder, exist_ok=True)

    train_example.to_csv(f"{example_folder}/train_m{example_m}_h{example_h}.csv", index=False)
    val_example.to_csv(f"{example_folder}/validation_m{example_m}_h{example_h}.csv", index=False)
    test_example.to_csv(f"{example_folder}/test_m{example_m}_h{example_h}.csv", index=False)

    print("Example datasets saved to:")
    print(example_folder)
else:
    print("Skipped example dataset saving because RUN_EXAMPLE_DATASET = False.")

Skipped example dataset saving because RUN_EXAMPLE_DATASET = False.


## Cell 16: Build all cascade non-sequential datasets efficiently

### Non-technical note for Cell 18 — save summary tables
This cell saves dataset-level summaries after generation. These files are not model inputs; they are checking files.

Use these summaries to answer important questions before modelling: how many windows were created, how many positive windows exist for each target, how many positive cases exist, and how many windows were removed by horizon censoring.

In [30]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

observation_windows = [4, 6, 12, 24]
prediction_horizons = [4, 6, 12]

split_names = ["train", "validation", "test"]

dataset_summary = []
missingness_summary = []
horizon_censoring_summary = []

# Reusable feature bases are saved because feature extraction is the expensive step.
# These internal files contain event timestamps and Movement End DT for later labelling.
# Do NOT use them directly for model training.
feature_base_dir = f"{output_dir}/feature_bases_internal"
os.makedirs(feature_base_dir, exist_ok=True)

# ------------------------------------------------------------
# Case split summary.
# This describes the train/validation/test split at case level.
# ------------------------------------------------------------
# "case_positive_*" means the case has the event label.
# "timed_positive_*" means the case has a usable event timestamp for
# window-level labelling.

for event_name, event_col in {
    "hd_icu": "event_time_hd_icu",
    "o2": "event_time_o2",
    "iv": "event_time_iv",
    "met": "event_time_met",
    "death": "event_time_death"
}.items():
    full_cohort[f"timed_positive_{event_name}"] = (
        full_cohort[event_col].notna().astype(int)
    )

full_cohort["timed_positive_any"] = (
    full_cohort[[
        "timed_positive_hd_icu",
        "timed_positive_o2",
        "timed_positive_iv",
        "timed_positive_met",
        "timed_positive_death"
    ]]
    .max(axis=1)
    .astype(int)
)

case_split_summary = (
    full_cohort
    .groupby("data_split")
    .agg(
        n_cases=("case_no_deid_clean", "nunique"),
        case_positive_any=("label_any_deterioration", "sum"),
        case_positive_hd_icu=("label_hd_icu", "sum"),
        case_positive_o2=("label_o2_increase", "sum"),
        case_positive_iv=("label_iv_increase", "sum"),
        case_positive_met=("label_met", "sum"),
        case_positive_death=("label_death", "sum"),
        timed_positive_any=("timed_positive_any", "sum"),
        timed_positive_hd_icu=("timed_positive_hd_icu", "sum"),
        timed_positive_o2=("timed_positive_o2", "sum"),
        timed_positive_iv=("timed_positive_iv", "sum"),
        timed_positive_met=("timed_positive_met", "sum"),
        timed_positive_death=("timed_positive_death", "sum")
    )
    .reset_index()
)

for col in [
    "case_positive_any",
    "case_positive_hd_icu",
    "case_positive_o2",
    "case_positive_iv",
    "case_positive_met",
    "case_positive_death",
    "timed_positive_any",
    "timed_positive_hd_icu",
    "timed_positive_o2",
    "timed_positive_iv",
    "timed_positive_met",
    "timed_positive_death"
]:
    case_split_summary[f"{col}_rate"] = (
        case_split_summary[col] / case_split_summary["n_cases"]
    )

case_split_summary_path = f"{output_dir}/case_split_summary.csv"
case_split_summary.to_csv(case_split_summary_path, index=False)

print("Case split summary:")
display(case_split_summary)

print("Saved case split summary to:")
print(case_split_summary_path)


# ------------------------------------------------------------
# Build datasets.
# Runtime improvement:
#   For each m and split, feature rows are built once and saved.
#   Then cascade labels are added separately for h = 4, 6, 12.
# ------------------------------------------------------------
for m in observation_windows:

    print("=" * 70)
    print(f"Building reusable cascade feature bases for m={m}h")
    print("=" * 70)

    feature_base_by_split = {}

    for split_name in split_names:
        feature_base = build_window_feature_base(
            case_table=case_features,
            vitals_by_case=vitals_by_case,
            m_hours=m,
            split_name=split_name
        )

        feature_base_by_split[split_name] = feature_base

        feature_base_path = f"{feature_base_dir}/{split_name}_feature_base_m{m}.csv"
        feature_base.to_csv(feature_base_path, index=False)

        print(f"Feature base {split_name}, m={m}: {feature_base.shape}")
        print(f"Saved reusable internal feature base to: {feature_base_path}")

    for h in prediction_horizons:

        print("-" * 70)
        print(f"Creating cascade-labelled datasets: m={m}h, h={h}h")
        print("-" * 70)

        current_folder = f"{output_dir}/m{m}_h{h}"
        os.makedirs(current_folder, exist_ok=True)

        for split_name in split_names:

            feature_base = feature_base_by_split[split_name]

            # Count windows removed by the full-horizon observability rule.
            horizon_censoring_summary.append(
                summarize_horizon_censoring(
                    feature_base_df=feature_base,
                    m_hours=m,
                    h_hours=h,
                    split_name=split_name
                )
            )

            split_df = add_horizon_labels(
                feature_base,
                h_hours=h
            )

            save_path = f"{current_folder}/{split_name}_m{m}_h{h}.csv"
            split_df.to_csv(save_path, index=False)

            summary_row = {
                "m_hours": m,
                "h_hours": h,
                "split": split_name,
                "n_windows": len(split_df),
                "n_cases": split_df["case_no_deid_clean"].nunique() if len(split_df) > 0 else 0
            }

            # Cascade labels may be NaN when the target event already happened
            # before window_end. Therefore, positive rates are calculated among
            # eligible windows only.
            for target in target_cols:
                eligible_col = f"eligible_{target}"

                if len(split_df) > 0:
                    eligible_mask = split_df[eligible_col].eq(1) if eligible_col in split_df.columns else split_df[target].notna()
                    eligible_windows = int(eligible_mask.sum())
                    eligible_cases = int(split_df.loc[eligible_mask, "case_no_deid_clean"].nunique())

                    positive_mask = split_df[target].eq(1)
                    positive_window_count = int(positive_mask.sum())
                    positive_case_count = int(split_df.loc[positive_mask, "case_no_deid_clean"].nunique())

                    summary_row[f"{target}_eligible_windows"] = eligible_windows
                    summary_row[f"{target}_eligible_cases"] = eligible_cases
                    summary_row[f"{target}_positive_windows"] = positive_window_count
                    summary_row[f"{target}_positive_window_rate_among_eligible"] = (
                        positive_window_count / eligible_windows if eligible_windows > 0 else np.nan
                    )
                    summary_row[f"{target}_positive_cases"] = positive_case_count
                    summary_row[f"{target}_positive_case_rate_among_eligible"] = (
                        positive_case_count / eligible_cases if eligible_cases > 0 else np.nan
                    )
                else:
                    summary_row[f"{target}_eligible_windows"] = 0
                    summary_row[f"{target}_eligible_cases"] = 0
                    summary_row[f"{target}_positive_windows"] = 0
                    summary_row[f"{target}_positive_window_rate_among_eligible"] = np.nan
                    summary_row[f"{target}_positive_cases"] = 0
                    summary_row[f"{target}_positive_case_rate_among_eligible"] = np.nan

            dataset_summary.append(summary_row)

            # Missingness summary.
            missing_flag_cols = [
                col for col in split_df.columns
                if col.endswith("_missing_flag")
            ]

            for col in missing_flag_cols:
                missingness_summary.append({
                    "m_hours": m,
                    "h_hours": h,
                    "split": split_name,
                    "missing_feature": col,
                    "missing_rate": split_df[col].mean() if len(split_df) > 0 else np.nan
                })

            print(f"{split_name}: {split_df.shape}")

        print("Saved folder:")
        print(current_folder)


Case split summary:


,data_split,n_cases,case_positive_any,case_positive_hd_icu,case_positive_o2,case_positive_iv,case_positive_met,case_positive_death,timed_positive_any,timed_positive_hd_icu,...,case_positive_o2_rate,case_positive_iv_rate,case_positive_met_rate,case_positive_death_rate,timed_positive_any_rate,timed_positive_hd_icu_rate,timed_positive_o2_rate,timed_positive_iv_rate,timed_positive_met_rate,timed_positive_death_rate
0,test,67,29,16,2,11,0,8,20,2,...,0.029851,0.164179,0.000000,0.119403,0.298507,0.029851,0.029851,0.164179,0.000000,0.119403
1,train,308,133,42,16,54,3,65,106,6,...,0.051948,0.175325,0.009740,0.211039,0.344156,0.019481,0.051948,0.175325,0.009740,0.211039
2,validation,66,28,11,2,6,1,15,20,2,...,0.030303,0.090909,0.015152,0.227273,0.303030,0.030303,0.030303,0.090909,0.015152,0.227273


Saved case split summary to:
/content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/case_split_summary.csv
Building reusable cascade feature bases for m=4h
Feature base train, m=4: (53203, 400)
Saved reusable internal feature base to: /content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/feature_bases_internal/train_feature_base_m4.csv
Feature base validation, m=4: (11575, 400)
Saved reusable internal feature base to: /content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/feature_bases_internal/validation_feature_base_m4.csv
Feature base test, m=4: (10193, 400)
Saved reusable internal feature base to: /content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/feature_bases_internal/test_feature_base_m4.csv
----------------------------------------------------------------------
Creating cascade-labelled datasets: m=4h, h=4h
-------------

## Cell 1 explanation — paths and switches

This cell connects to Google Drive, defines input/output file paths, and controls optional behaviours.

Important switches:

- `RUN_EXAMPLE_DATASET`: if `False`, skip the example dataset build to save time.
- Output folder: stores final v8 datasets and summary tables.

The notebook should be run in Google Colab because it expects files under `/content/drive/My Drive/...`.


## Cell 17: Save summary of all generated datasets

### Non-technical note for Cell 19 — quick quality check
This cell reloads generated datasets and prints quick checks. It helps confirm that output files were saved correctly and that target columns have reasonable positive counts.

This is a safety check before moving to modelling.

In [31]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

dataset_summary_df = pd.DataFrame(dataset_summary)

dataset_summary_path = f"{output_dir}/dataset_generation_summary.csv"
dataset_summary_df.to_csv(dataset_summary_path, index=False)

print("Dataset generation summary:")
display(dataset_summary_df)

print("Saved dataset summary to:")
print(dataset_summary_path)


missingness_summary_df = pd.DataFrame(missingness_summary)

missingness_summary_path = f"{output_dir}/missingness_summary.csv"
missingness_summary_df.to_csv(missingness_summary_path, index=False)

print("\nMissingness summary preview:")
display(missingness_summary_df.head(20))

print("Saved missingness summary to:")
print(missingness_summary_path)


horizon_censoring_summary_df = pd.DataFrame(horizon_censoring_summary)

horizon_censoring_summary_path = f"{output_dir}/horizon_censoring_summary.csv"
horizon_censoring_summary_df.to_csv(horizon_censoring_summary_path, index=False)

print("\nHorizon-censoring summary:")
display(horizon_censoring_summary_df)

print("Saved horizon-censoring summary to:")
print(horizon_censoring_summary_path)


Dataset generation summary:


,m_hours,h_hours,split,n_windows,n_cases,y_any_eligible_windows,y_any_eligible_cases,y_any_positive_windows,y_any_positive_window_rate_among_eligible,y_any_positive_cases,...,y_met_positive_windows,y_met_positive_window_rate_among_eligible,y_met_positive_cases,y_met_positive_case_rate_among_eligible,y_death_eligible_windows,y_death_eligible_cases,y_death_positive_windows,y_death_positive_window_rate_among_eligible,y_death_positive_cases,y_death_positive_case_rate_among_eligible
0,4,4,train,51971,308,51971,308,236,0.004541,50,...,9,0.000177,3,0.009740,51971,308,0,0.0,0,0.0
1,4,4,validation,11311,66,11311,66,25,0.002210,7,...,4,0.000354,1,0.015152,11311,66,0,0.0,0,0.0
2,4,4,test,9925,67,9925,67,52,0.005239,13,...,0,0.000000,0,0.000000,9925,67,0,0.0,0,0.0
3,4,6,train,51355,307,51355,307,345,0.006718,50,...,13,0.000258,3,0.009772,51355,307,0,0.0,0,0.0
4,4,6,validation,11179,66,11179,66,37,0.003310,7,...,6,0.000537,1,0.015152,11179,66,0,0.0,0,0.0
5,4,6,test,9791,67,9791,67,76,0.007762,13,...,0,0.000000,0,0.000000,9791,67,0,0.0,0,0.0
6,4,12,train,49515,305,49515,305,639,0.012905,50,...,25,0.000516,3,0.009836,49515,305,0,0.0,0,0.0
7,4,12,validation,10783,66,10783,66,69,0.006399,7,...,12,0.001113,1,0.015152,10783,66,0,0.0,0,0.0
8,4,12,test,9389,67,9389,67,140,0.014911,13,...,0,0.000000,0,0.000000,9389,67,0,0.0,0,0.0
9,6,4,train,51355,307,51355,307,235,0.004576,50,...,9,0.000179,3,0.009772,51355,307,0,0.0,0,0.0


Saved dataset summary to:
/content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/dataset_generation_summary.csv

Missingness summary preview:


,m_hours,h_hours,split,missing_feature,missing_rate
0,4,4,train,DBP_missing_flag,0.083893
1,4,4,train,HR_missing_flag,0.077312
2,4,4,train,RR_missing_flag,0.052087
3,4,4,train,SBP_missing_flag,0.083893
4,4,4,train,SpO2_missing_flag,0.078217
5,4,4,train,skin_temperature_missing_flag,0.566066
6,4,4,train,temperature_missing_flag,0.050990
7,4,4,train,baseline_DBP_missing_flag,0.472360
8,4,4,train,baseline_HR_missing_flag,0.434569
9,4,4,train,baseline_RR_missing_flag,0.287718


Saved missingness summary to:
/content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/missingness_summary.csv

Horizon-censoring summary:


,m_hours,h_hours,split,windows_before_horizon_filter,windows_after_horizon_filter,removed_windows,removed_rate,censoring_rule
0,4,4,train,53203,51971,1232,0.023157,Retain if horizon_end <= Movement End DT; remo...
1,4,4,validation,11575,11311,264,0.022808,Retain if horizon_end <= Movement End DT; remo...
2,4,4,test,10193,9925,268,0.026293,Retain if horizon_end <= Movement End DT; remo...
3,4,6,train,53203,51355,1848,0.034735,Retain if horizon_end <= Movement End DT; remo...
4,4,6,validation,11575,11179,396,0.034212,Retain if horizon_end <= Movement End DT; remo...
5,4,6,test,10193,9791,402,0.039439,Retain if horizon_end <= Movement End DT; remo...
6,4,12,train,53203,49515,3688,0.069319,Retain if horizon_end <= Movement End DT; remo...
7,4,12,validation,11575,10783,792,0.068423,Retain if horizon_end <= Movement End DT; remo...
8,4,12,test,10193,9389,804,0.078878,Retain if horizon_end <= Movement End DT; remo...
9,6,4,train,52587,51355,1232,0.023428,Retain if horizon_end <= Movement End DT; remo...


Saved horizon-censoring summary to:
/content/drive/My Drive/merged_240920_240822_prepared/nonsequential_window_dataset_v8_cascade/horizon_censoring_summary.csv


## Cell 1 explanation — paths and switches

This cell connects to Google Drive, defines input/output file paths, and controls optional behaviours.

Important switches:

- `RUN_EXAMPLE_DATASET`: if `False`, skip the example dataset build to save time.
- Output folder: stores final v8 datasets and summary tables.

The notebook should be run in Google Colab because it expects files under `/content/drive/My Drive/...`.


## Cell 18: Quick quality check for generated datasets

### Non-technical note for Cell 19B — feature-set helper
This cell defines how to select model input columns from the full generated dataset. It does not rebuild datasets. It only filters columns.

This design saves time. You build each full dataset once, then compare different feature sets by selecting different column groups, such as baseline patient+vital features, secondary diagnosis features, last60min features, or cascade prior-event features.

In [32]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

for _, row in dataset_summary_df.iterrows():

    m = int(row["m_hours"])
    h = int(row["h_hours"])
    split_name = row["split"]

    folder = f"{output_dir}/m{m}_h{h}"
    data_path = f"{folder}/{split_name}_m{m}_h{h}.csv"

    temp_df = pd.read_csv(data_path)

    print("=" * 60)
    print(f"Checking m={m}, h={h}, split={split_name}")
    print("Rows:", temp_df.shape[0])
    print("Columns:", temp_df.shape[1])
    print("Unique cases:", temp_df["case_no_deid_clean"].nunique())

    print("\nWindow-level positive counts:")
    print(temp_df[target_cols].sum())

    print("\nWindow-level positive rates:")
    print(temp_df[target_cols].mean())

    missing_flag_cols = [
        col for col in temp_df.columns
        if col.endswith("_missing_flag")
    ]

    if len(missing_flag_cols) > 0:
        print("\nAverage missingness rate across missing-flag features:")
        print(temp_df[missing_flag_cols].mean().mean())


Checking m=4, h=4, split=train
Rows: 51971
Columns: 408
Unique cases: 308

Window-level positive counts:
y_any       236.0
y_hd_icu      0.0
y_o2         60.0
y_iv        171.0
y_met         9.0
y_death       0.0
dtype: float64

Window-level positive rates:
y_any       0.004541
y_hd_icu    0.000000
y_o2        0.001234
y_iv        0.004041
y_met       0.000177
y_death     0.000000
dtype: float64

Average missingness rate across missing-flag features:
0.33287978368888876
Checking m=4, h=4, split=validation
Rows: 11311
Columns: 408
Unique cases: 66

Window-level positive counts:
y_any       25.0
y_hd_icu     0.0
y_o2         5.0
y_iv        16.0
y_met        4.0
y_death      0.0
dtype: float64

Window-level positive rates:
y_any       0.002210
y_hd_icu    0.000000
y_o2        0.000454
y_iv        0.001601
y_met       0.000354
y_death     0.000000
dtype: float64

Average missingness rate across missing-flag features:
0.34174907696258594
Checking m=4, h=4, split=test
Rows: 9925
Columns: 40

## Cell 1 explanation — paths and switches

This cell connects to Google Drive, defines input/output file paths, and controls optional behaviours.

Important switches:

- `RUN_EXAMPLE_DATASET`: if `False`, skip the example dataset build to save time.
- Output folder: stores final v8 datasets and summary tables.

The notebook should be run in Google Colab because it expects files under `/content/drive/My Drive/...`.


## Cell 19: Helper for leakage-safe cascade feature-set comparison

### Non-technical note for Cell 20 — first-deterioration subset inside v8
The v8 dataset is cascade-capable, but it also contains first-deterioration prediction rows.

Rows with `prior_event_count == 0` are windows before any deterioration event has occurred. This cell provides helper functions to filter those rows and prepare first-deterioration models from the same generated v8 dataset.

In [33]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# ============================================================
# Cell 19: Helper for leakage-safe cascade feature-set comparison
# ============================================================
# Use this after loading one generated train/validation/test CSV.
#
# Design:
#   Build the full dataset once, then compare feature sets by selecting
#   different column groups from the same CSV.
#
# Cascade modelling note:
#   Use prepare_model_data(...) so ineligible windows are removed for the
#   selected target. For example, y_o2 should not be trained on windows
#   after O2 increase already happened.
# ============================================================

metadata_cols = [
    "case_no_deid_clean", "data_split", "m_hours", "h_hours",
    "window_start", "window_end", "horizon_end"
]

target_cols = ["y_any", "y_hd_icu", "y_o2", "y_iv", "y_met", "y_death"]
eligibility_cols = [f"eligible_{target}" for target in target_cols]

patient_model_cols = [
    "Age", "Gender", "Race", "Admit Type Description",
    "data_batch", "hours_since_w36_start"
]


def get_feature_sets(df):
    """
    Return leakage-safe feature sets for model comparison.

    Included by design:
        - prior-event features are valid cascade predictors because they
          happened before window_end.
        - baseline/recent contrast features are vital-sign information.

    Excluded by design:
        - target columns
        - eligibility columns, because they are filtering indicators
        - metadata columns
    """

    dx_cols = [
        col for col in df.columns
        if col.startswith("secondary_dx_") or col == "secondary_diagnosis_count"
    ]

    last60_cols = [col for col in df.columns if col.startswith("last60min_")]
    last30_cols = [col for col in df.columns if col.startswith("last30min_")]

    event_history_cols = [
        col for col in df.columns
        if col.startswith("prior_")
        or col.startswith("time_since_prior_")
        or col in [
            "prior_event_count",
            "any_prior_deterioration_flag",
            "time_since_first_prior_event_hours",
            "time_since_most_recent_prior_event_hours"
        ]
    ]

    excluded_cols = set(
        metadata_cols + target_cols + eligibility_cols +
        dx_cols + last60_cols + last30_cols + event_history_cols
    )

    # Vital columns include:
    #   full-window features: HR_mean, RR_slope, ...
    #   baseline/recent features: baseline_HR_mean, recent_HR_mean, ...
    #   contrast features: recent_minus_baseline_HR_mean, ...
    #   vital availability features: available_vital_type_count, ...
    vital_cols = [
        col for col in df.columns
        if col not in excluded_cols
        and col not in patient_model_cols
    ]

    baseline_patient_vital = patient_model_cols + vital_cols
    cascade_patient_vital = baseline_patient_vital + event_history_cols

    return {
        "baseline_patient_vital": baseline_patient_vital,
        "cascade_patient_vital_history": cascade_patient_vital,
        "cascade_plus_secondary_dx": cascade_patient_vital + dx_cols,
        "cascade_plus_last60min": cascade_patient_vital + last60_cols,
        "cascade_plus_last30min": cascade_patient_vital + last30_cols,
        "cascade_final60_dx_last60": cascade_patient_vital + dx_cols + last60_cols,
        "cascade_final30_dx_last30": cascade_patient_vital + dx_cols + last30_cols,
        "cascade_full_optional_dx_last60_last30": cascade_patient_vital + dx_cols + last60_cols + last30_cols
    }


def prepare_model_data(df, target_col, feature_set_name):
    """
    Prepare X and y for one target and one feature set.

    This function removes ineligible windows automatically.

    Example:
        X, y, feature_cols = prepare_model_data(
            train_df,
            target_col="y_o2",
            feature_set_name="cascade_final60_dx_last60"
        )
    """

    feature_sets = get_feature_sets(df)

    if feature_set_name not in feature_sets:
        raise ValueError(
            f"Unknown feature_set_name: {feature_set_name}. "
            f"Available options: {list(feature_sets.keys())}"
        )

    if target_col not in target_cols:
        raise ValueError(f"target_col should be one of: {target_cols}")

    eligible_col = f"eligible_{target_col}"

    model_df = df.copy()

    if eligible_col in model_df.columns:
        model_df = model_df[model_df[eligible_col] == 1].copy()

    # Also remove rows with missing target values.
    model_df = model_df[model_df[target_col].notna()].copy()

    feature_cols = feature_sets[feature_set_name]
    feature_cols = [col for col in feature_cols if col in model_df.columns]

    X = model_df[feature_cols].copy()
    y = model_df[target_col].astype(int).copy()

    return X, y, feature_cols


print("Cascade feature-set helper loaded.")
print("Use get_feature_sets(df) to inspect columns.")
print("Use prepare_model_data(df, target_col, feature_set_name) before modelling.")


Cascade feature-set helper loaded.
Use get_feature_sets(df) to inspect columns.
Use prepare_model_data(df, target_col, feature_set_name) before modelling.


# Cell 20: How to use v8 for first-deterioration prediction

The v8 dataset is cascade-capable, but first-deterioration prediction still exists inside it.

## Cascade modelling

Use the full v8 dataset.  
For each target, keep only rows where that target is still eligible.

Example:

- If O2 already happened before `window_end`, then `eligible_y_o2 = 0` and this row should not train the O2 model.
- The same row may still be valid for MET, HD/ICU, or death if those events have not happened yet.

## First-deterioration modelling

Use only rows before any deterioration has already happened:

```python
prior_event_count == 0
```

In this subset, the model asks:

> “Before any deterioration event has happened, can we predict the first upcoming deterioration event?”

This gives you both modelling strategies without rebuilding separate datasets.


In [34]:
# ------------------------------------------------------------
# Reader note:
# The markdown cell above explains this code cell in plain language.
# The code below keeps the same dataset-building logic; comments are added
# only to make the workflow easier to audit and understand.
# ------------------------------------------------------------

# ============================================================
# Cell 20: Helper for first-deterioration prediction inside v8
# ============================================================
# The v8 dataset is mainly cascade-capable.
# However, the first-deterioration setting is already inside it.
#
# First-deterioration windows are rows where no prior deterioration
# event has happened before window_end.
#
# These rows answer:
#   “Before any deterioration has happened, can we predict whether
#    deterioration will happen in the next h hours?”
# ============================================================


def filter_first_deterioration_windows(df):
    """
    Keep only windows before any deterioration event has happened.

    This is the first-deterioration prediction subset.

    Required columns:
        prior_event_count
        any_prior_deterioration_flag
    """

    required_cols = ["prior_event_count", "any_prior_deterioration_flag"]

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(
            "The dataset does not contain required first-deterioration "
            f"columns: {missing_cols}"
        )

    first_df = df[
        (df["prior_event_count"] == 0) &
        (df["any_prior_deterioration_flag"] == 0)
    ].copy()

    return first_df


def get_first_deterioration_feature_sets(df):
    """
    Return feature sets for first-deterioration modelling.

    Difference from cascade feature sets:
        We remove prior-event history features because they are all zero or
        empty in the first-deterioration subset.
    """

    cascade_sets = get_feature_sets(df)

    event_history_cols = [
        col for col in df.columns
        if col.startswith("prior_")
        or col.startswith("time_since_prior_")
        or col in [
            "prior_event_count",
            "any_prior_deterioration_flag",
            "time_since_first_prior_event_hours",
            "time_since_most_recent_prior_event_hours"
        ]
    ]

    def remove_event_history(cols):
        return [col for col in cols if col not in event_history_cols]

    return {
        "first_patient_vital": remove_event_history(
            cascade_sets["baseline_patient_vital"]
        ),
        "first_plus_secondary_dx": remove_event_history(
            cascade_sets["cascade_plus_secondary_dx"]
        ),
        "first_plus_last60min": remove_event_history(
            cascade_sets["cascade_plus_last60min"]
        ),
        "first_plus_last30min": remove_event_history(
            cascade_sets["cascade_plus_last30min"]
        ),
        "first_final60_dx_last60": remove_event_history(
            cascade_sets["cascade_final60_dx_last60"]
        ),
        "first_final30_dx_last30": remove_event_history(
            cascade_sets["cascade_final30_dx_last30"]
        ),
        "first_full_optional_dx_last60_last30": remove_event_history(
            cascade_sets["cascade_full_optional_dx_last60_last30"]
        )
    }


def prepare_first_deterioration_model_data(df, target_col, feature_set_name):
    """
    Prepare X and y for first-deterioration prediction.

    Steps:
        1. Keep only pre-first-deterioration windows.
        2. Apply the normal target eligibility rule.
        3. Select one requested feature set.
    """

    first_df = filter_first_deterioration_windows(df)
    feature_sets = get_first_deterioration_feature_sets(first_df)

    if feature_set_name not in feature_sets:
        raise ValueError(
            f"Unknown feature_set_name: {feature_set_name}. "
            f"Available options: {list(feature_sets.keys())}"
        )

    if target_col not in target_cols:
        raise ValueError(f"target_col should be one of: {target_cols}")

    eligible_col = f"eligible_{target_col}"

    model_df = first_df.copy()

    if eligible_col in model_df.columns:
        model_df = model_df[model_df[eligible_col] == 1].copy()

    model_df = model_df[model_df[target_col].notna()].copy()

    feature_cols = feature_sets[feature_set_name]
    feature_cols = [col for col in feature_cols if col in model_df.columns]

    X = model_df[feature_cols].copy()
    y = model_df[target_col].astype(int).copy()

    return X, y, feature_cols


SAVE_FIRST_DETERIORATION_SUBSETS = False

print("First-deterioration helper loaded.")
print("Use filter_first_deterioration_windows(df) to get pre-first-event rows.")
print("Use prepare_first_deterioration_model_data(...) before modelling first deterioration.")


First-deterioration helper loaded.
Use filter_first_deterioration_windows(df) to get pre-first-event rows.
Use prepare_first_deterioration_model_data(...) before modelling first deterioration.
